# Point Cloud Classification Project, 

### Multi-Model Kitchen Sink with Ground Proxy, Spatial CV, Component Cleanup

Josh Brewster (but vibe coded in GPT 5.4 Thinking)

## Key design choices in this notebook 

This notebook keeps a non-neural, feature-engineered workflow, but folds in three upgrades that are especially useful for the error modes you called out:

- **Dedicated ground-surface proxy features** built from a low-envelope support grid with morphological smoothing. This is still lighter than a full CSF package, but it is much closer in spirit to an upstream terrain model than relying only on neighborhood minima.
- **Spatial / blockwise validation** alongside the usual random point-wise CV, so you can see the more realistic score when nearby points are forced to stay together.
- **Connected-component cleanup** on prediction grids so tiny building and car islands can be relabeled object-wise instead of purely point-wise.

GPU notes:
- XGBoost can use CUDA when your local build already supports it
- cuML random forest and nearest-neighbor search are only used when they import successfully

The goal is still a strong classical baseline, but a more honest and more spatially coherent one.


## Block summary: environment and package setup

Methods used in this block:
- install the core Python stack used by the notebook
- keep optional GPU libraries out of the automatic install path, because RAPIDS and CUDA wheels are environment-specific
- allow the rest of the notebook to run on CPU even when GPU libraries are absent

Notes:
- XGBoost GPU and cuML GPU paths are supported later in the notebook, but they are **optional** and guarded by runtime checks rather than forced installs. citeturn776407view0turn776407view1


In [1]:
%pip -q install laspy scipy scikit-learn xgboost joblib tqdm pandas matplotlib pillow

Note: you may need to restart the kernel to use updated packages.


## Block summary: imports, configuration, labels, and run switches

Methods used in this block:
- import the classical ML, LiDAR, and plotting stack
- expose toggles for adaptive geometry features
- expose toggles for contextual kNN probability smoothing
- expose switches for a grid-derived ground proxy, spatial CV, and connected-component cleanup
- expose optional GPU toggles for XGBoost, cuML random forest, and cuML nearest-neighbor search

Why these are here:
- the biggest classical improvements still available after the earlier notebook versions are better terrain normalization, more realistic spatial validation, and a light object-wise cleanup pass


In [2]:
from __future__ import annotations

import copy
import gc
import json
import math
import time
from itertools import product
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import joblib
import laspy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.stats import skew
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAVE_STRATIFIED_GROUP_KFOLD = True
except Exception:
    StratifiedGroupKFold = None
    HAVE_STRATIFIED_GROUP_KFOLD = False
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm.auto import tqdm

try:
    from xgboost import XGBClassifier
    HAVE_XGBOOST = True
except Exception:
    HAVE_XGBOOST = False

try:
    import cupy as cp
    HAVE_CUPY = True
except Exception:
    HAVE_CUPY = False

try:
    from cuml.ensemble import RandomForestClassifier as cuRFClassifier
    from cuml.neighbors import NearestNeighbors as cuNearestNeighbors
    HAVE_CUML = True
except Exception:
    HAVE_CUML = False

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

RANDOM_STATE = 42

ROOT = Path(".")
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

OUTPUT_DIR = ROOT / "outputs_multimodel_kitchensink_learned"
CACHE_DIR = OUTPUT_DIR / "cache"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"
PRED_DIR = OUTPUT_DIR / "predictions"
ARRAY_DIR = CACHE_DIR / "arrays"

for directory in [OUTPUT_DIR, CACHE_DIR, MODEL_DIR, REPORT_DIR, PRED_DIR, ARRAY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Buildings", "Trees", "Ground", "Cars"]
CLASS_TO_LABEL = {name.lower(): i for i, name in enumerate(CLASS_NAMES)}
LABEL_TO_CLASS = {i: name for name, i in CLASS_TO_LABEL.items()}
OUTPUT_CLASS_CODE_OFFSET = 1

MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

CHUNK_READ_POINTS = 1_000_000
SUPPORT_GRID_SIZE = 1.0
GRID_SIZES = [1.0, 2.0, 4.0, 8.0, 16.0]
SUPPORT_KS = [8, 16, 32]
SUPPORT_GEOM_KS = [16, 32]
SUPPORT_ADAPTIVE_GEOM_KS = [8, 16, 32, 64]
SUPPORT_TERRAIN_KS = [16, 32, 64]
SUPPORT_MAX_CELLS = 2_000_000

DROP_TRAIN_OUTLIERS = True
TRAIN_OUTLIER_FEATURE = "support_mean_d_k16"
TRAIN_OUTLIER_Q = 0.995
DROP_TEST_OUTLIERS_WITH_TRAIN_THRESHOLD = False
ALLOW_EMPTY_TEST_SPLIT = False

FEATURE_SELECTION_MAX_ROWS = 180_000
PERMUTATION_MAX_ROWS = 40_000
PERMUTATION_REPEATS = 2
RUN_PERMUTATION_IMPORTANCE = False

MAX_FEATURES_AFTER_CORR = 240
TOP_MI = 144
TOP_ET = 104
TOP_XGB = 104
TOP_PERM = 32
TOP_CONSENSUS = 56

CV_FOLDS = 3
CALIBRATION_FRACTION = 0.15
MODEL_NAMES_TO_RUN = [
    "ExtraTrees",
    "ExtraTreesRegularized",
    "ExtraTreesConservative",
    "ExtraTreesSoftVoteTuned",
    "HierarchicalExtraTreesTuned",
    "RandomForest",
    "XGBoost",
]
MODEL_FIT_SAMPLE_CAPS = {}
RUN_FULL_CLOUD_PREDICTION = True
FULL_PRED_BATCH_SIZE = 50_000
FORCE_REBUILD_ORIGINAL_CACHE = False
FORCE_REBUILD_FEATURES = True
FULL_PRED_BUILD_ONLY_SELECTED = True

GROUND_PROXY_GRID_SIZE = 1.0
GROUND_PROXY_OPENING_SIZE = 5
GROUND_PROXY_MEDIAN_SIZE = 3
GROUND_PROXY_SMOOTH_SIGMA = 1.0
GROUND_PROXY_CACHE_NAME = "ground_proxy_artifact.joblib"

RUN_CONTEXTUAL_POSTPROCESS = True
CONTEXT_K_GRID = [8, 16, 32]
CONTEXT_ALPHA_GRID = [0.20, 0.35, 0.50]
CONTEXT_SPATIAL_COLUMNS = ("x", "y", "z")
CONTEXT_SPATIAL_SCALES = (1.0, 1.0, 2.0)
CONTEXT_USE_DISTANCE_WEIGHTS = True
CONTEXT_SUFFIX = "ContextKNN"

RUN_SPATIAL_CV = True
SPATIAL_CV_FOLDS = 3
SPATIAL_CV_BLOCK_SIZE = 12.0

RUN_COMPONENT_CLEANUP = True
COMPONENT_GRID_SIZE = 1.0
COMPONENT_CONNECTIVITY = 8
COMPONENT_MIN_CELLS_BY_CLASS = {0: 6, 3: 2}
COMPONENT_SUFFIX = "ComponentClean"

USE_GPU_XGBOOST_IF_AVAILABLE = True
USE_CUML_RF_IF_AVAILABLE = True
USE_GPU_NEIGHBOR_SEARCH_IF_AVAILABLE = True

MATCH_NEAREST_FALLBACK = True
MATCH_NEAREST_TOLERANCE = 0.25
MATCH_NEAREST_SAMPLE_BATCH = 50_000
CACHE_VERSION = "multimodel_kitchensink_v2"
FEATURE_CACHE_TAG = "geomterrain_v7_ground_spatial_cleanup_gpu"

TEST_OUTPUT_DIR = OUTPUT_DIR / "test_predictions"
FIG_DIR = REPORT_DIR / "figures"
for directory in [TEST_OUTPUT_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print({
    "HAVE_XGBOOST": HAVE_XGBOOST,
    "HAVE_CUPY": HAVE_CUPY,
    "HAVE_CUML": HAVE_CUML,
    "USE_GPU_XGBOOST_IF_AVAILABLE": USE_GPU_XGBOOST_IF_AVAILABLE,
    "USE_CUML_RF_IF_AVAILABLE": USE_CUML_RF_IF_AVAILABLE,
    "USE_GPU_NEIGHBOR_SEARCH_IF_AVAILABLE": USE_GPU_NEIGHBOR_SEARCH_IF_AVAILABLE,
    "HAVE_STRATIFIED_GROUP_KFOLD": HAVE_STRATIFIED_GROUP_KFOLD,
    "RUN_SPATIAL_CV": RUN_SPATIAL_CV,
    "RUN_COMPONENT_CLEANUP": RUN_COMPONENT_CLEANUP,
})

{'HAVE_XGBOOST': True, 'HAVE_CUPY': False, 'HAVE_CUML': False, 'USE_GPU_XGBOOST_IF_AVAILABLE': True, 'USE_CUML_RF_IF_AVAILABLE': True, 'USE_GPU_NEIGHBOR_SEARCH_IF_AVAILABLE': True, 'HAVE_STRATIFIED_GROUP_KFOLD': True, 'RUN_SPATIAL_CV': True, 'RUN_COMPONENT_CLEANUP': True}


c:\Users\brews\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Block summary: original LAS loading, dimension discovery, and sample rematching

**Methods used here**

- locate the original LAS file
- read XYZ and available intrinsic dimensions safely
- match train/test sample points back to the original cloud
- use an exact match first, then a nearest-XYZ fallback when needed

**Why this block exists**

Point-wise supervision is only useful if the sampled labels are aligned correctly to the original cloud. Rematching is a practical but important step because even small coordinate mismatches can silently poison feature extraction and evaluation.


In [3]:
def find_original_las(root: Path = ROOT) -> Path:
    exact_candidates = [
        root / "original_point_cloud.las",
        root / "Original_Point_Cloud.las",
        root / "original_Point_Cloud.las",
        root / "Original_point_cloud.las",
    ]
    for path in exact_candidates:
        if path.exists():
            return path

    las_files = sorted(root.glob("*.las"))
    for path in las_files:
        stem = path.stem.lower()
        if "original" in stem and "point" in stem and "cloud" in stem:
            return path

    if len(las_files) == 1:
        return las_files[0]

    raise FileNotFoundError("Could not find the original LAS file at the notebook root.")


def class_name_from_path(path: Path) -> str:
    stem = path.stem.lower()
    for class_name in CLASS_NAMES:
        if class_name.lower() in stem:
            return class_name
    raise ValueError(f"Could not infer class from filename: {path.name}")


def get_dimension_name_from_names(dim_names: Sequence[str], target: str) -> Optional[str]:
    target = target.lower()
    for name in dim_names:
        if str(name).lower() == target:
            return str(name)
    return None


def chunk_dimension_names(points) -> List[str]:
    fmt = points.point_format
    try:
        return [str(name) for name in fmt.dimension_names]
    except Exception:
        return [dim.name for dim in fmt.dimensions]


def get_chunk_dim(points, name: str) -> Optional[np.ndarray]:
    actual = get_dimension_name_from_names(chunk_dimension_names(points), name)
    if actual is None:
        return None
    arr = np.asarray(points[actual])
    return arr


def get_raw_xyz_from_points(points) -> np.ndarray:
    return np.column_stack([
        np.asarray(points.X),
        np.asarray(points.Y),
        np.asarray(points.Z),
    ]).astype(np.int64, copy=False)


def get_scaled_xyz_from_points(points, scales: np.ndarray, offsets: np.ndarray) -> np.ndarray:
    raw = get_raw_xyz_from_points(points).astype(np.float64, copy=False)
    xyz = raw * scales.reshape(1, 3) + offsets.reshape(1, 3)
    return xyz.astype(np.float32, copy=False)


def load_sample_metadata(folder: Path) -> pd.DataFrame:
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")

    rows: List[pd.DataFrame] = []
    las_files = sorted(folder.glob("*.las"))
    if not las_files:
        raise FileNotFoundError(f"No LAS files found in {folder}")

    for las_path in las_files:
        class_name = class_name_from_path(las_path)
        label = CLASS_TO_LABEL[class_name.lower()]
        las = laspy.read(las_path)

        dim_names = [dim.name for dim in las.point_format.dimensions]
        id_name = get_dimension_name_from_names(dim_names, "Id")
        if id_name is None:
            raise KeyError(f"Sample LAS file is missing Id dimension: {las_path}")

        ids = np.asarray(las[id_name]).astype(np.int64, copy=False)
        raw_xyz = np.column_stack([
            np.asarray(las.X),
            np.asarray(las.Y),
            np.asarray(las.Z),
        ]).astype(np.int64, copy=False)
        xyz = np.column_stack([
            np.asarray(las.x),
            np.asarray(las.y),
            np.asarray(las.z),
        ]).astype(np.float32, copy=False)

        file_rows = pd.DataFrame(
            {
                "Id": ids,
                "X_raw": raw_xyz[:, 0],
                "Y_raw": raw_xyz[:, 1],
                "Z_raw": raw_xyz[:, 2],
                "x": xyz[:, 0],
                "y": xyz[:, 1],
                "z": xyz[:, 2],
                "label": np.int32(label),
                "class_name": class_name,
                "source_file": las_path.name,
                "row_in_file": np.arange(len(ids), dtype=np.int64),
            }
        )
        file_rows["sample_uid"] = [f"{las_path.name}:{i}" for i in range(len(file_rows))]
        rows.append(file_rows)

    return pd.concat(rows, ignore_index=True)


def stratified_cap_dataframe(df: pd.DataFrame, label_col: str, max_rows: Optional[int], random_state: int = RANDOM_STATE) -> pd.DataFrame:
    if max_rows is None or len(df) <= max_rows:
        return df.reset_index(drop=True).copy()

    rng = np.random.default_rng(random_state)
    total = len(df)
    picked: List[np.ndarray] = []

    for _, group in df.groupby(label_col, sort=False):
        target = max(1, int(round(len(group) * max_rows / total)))
        target = min(target, len(group))
        chosen = rng.choice(group.index.to_numpy(), size=target, replace=False)
        picked.append(np.sort(chosen))

    picked_idx = np.concatenate(picked) if picked else np.empty(0, dtype=np.int64)
    out = df.loc[picked_idx].copy()

    if len(out) > max_rows:
        out = out.sample(n=max_rows, random_state=random_state)
    elif len(out) < max_rows:
        remaining = df.drop(index=out.index)
        need = min(max_rows - len(out), len(remaining))
        if need > 0:
            extra = remaining.sample(n=need, random_state=random_state)
            out = pd.concat([out, extra], ignore_index=False)

    return out.sort_index().reset_index(drop=True)


def make_id_xyz_key_array(ids: np.ndarray, raw_xyz: np.ndarray) -> np.ndarray:
    keys = np.empty(
        len(ids),
        dtype=[("Id", np.int64), ("X", np.int64), ("Y", np.int64), ("Z", np.int64)],
    )
    keys["Id"] = np.asarray(ids, dtype=np.int64)
    keys["X"] = raw_xyz[:, 0]
    keys["Y"] = raw_xyz[:, 1]
    keys["Z"] = raw_xyz[:, 2]
    return keys


def make_xyz_key_array(raw_xyz: np.ndarray) -> np.ndarray:
    keys = np.empty(
        len(raw_xyz),
        dtype=[("X", np.int64), ("Y", np.int64), ("Z", np.int64)],
    )
    keys["X"] = raw_xyz[:, 0]
    keys["Y"] = raw_xyz[:, 1]
    keys["Z"] = raw_xyz[:, 2]
    return keys


def init_sorted_sample_key_index(sample_df: pd.DataFrame) -> Dict[str, np.ndarray]:
    raw_xyz = sample_df[["X_raw", "Y_raw", "Z_raw"]].to_numpy(np.int64, copy=True)
    ids = sample_df["Id"].to_numpy(np.int64, copy=True)

    idxyz = make_id_xyz_key_array(ids, raw_xyz)
    idxyz_sorter = np.argsort(idxyz, order=("Id", "X", "Y", "Z"), kind="mergesort")
    xyz = make_xyz_key_array(raw_xyz)
    xyz_sorter = np.argsort(xyz, order=("X", "Y", "Z"), kind="mergesort")

    return {
        "idxyz_keys_sorted": idxyz[idxyz_sorter],
        "idxyz_sorter": idxyz_sorter.astype(np.int64, copy=False),
        "xyz_keys_sorted": xyz[xyz_sorter],
        "xyz_sorter": xyz_sorter.astype(np.int64, copy=False),
    }


def assign_matches_from_chunk(
    chunk_global_indices: np.ndarray,
    chunk_raw_xyz: np.ndarray,
    chunk_ids: np.ndarray,
    sample_index: Dict[str, np.ndarray],
    out_matches: np.ndarray,
    method_out: np.ndarray,
) -> None:
    unresolved = np.where(out_matches < 0)[0]
    if len(unresolved) == 0:
        return

    # Pass 1: Id + raw XYZ
    query_idxyz = make_id_xyz_key_array(chunk_ids, chunk_raw_xyz)
    left = np.searchsorted(sample_index["idxyz_keys_sorted"], query_idxyz, side="left")
    right = np.searchsorted(sample_index["idxyz_keys_sorted"], query_idxyz, side="right")

    for chunk_row in np.where(left < right)[0]:
        candidates = sample_index["idxyz_sorter"][left[chunk_row]:right[chunk_row]]
        unresolved_candidates = candidates[out_matches[candidates] < 0]
        if len(unresolved_candidates) == 0:
            continue
        sample_row = int(unresolved_candidates[0])
        out_matches[sample_row] = int(chunk_global_indices[chunk_row])
        method_out[sample_row] = "id+raw_xyz"

    unresolved = np.where(out_matches < 0)[0]
    if len(unresolved) == 0:
        return

    # Pass 2: raw XYZ exact
    query_xyz = make_xyz_key_array(chunk_raw_xyz)
    left = np.searchsorted(sample_index["xyz_keys_sorted"], query_xyz, side="left")
    right = np.searchsorted(sample_index["xyz_keys_sorted"], query_xyz, side="right")

    for chunk_row in np.where(left < right)[0]:
        candidates = sample_index["xyz_sorter"][left[chunk_row]:right[chunk_row]]
        unresolved_candidates = candidates[out_matches[candidates] < 0]
        if len(unresolved_candidates) == 0:
            continue
        sample_row = int(unresolved_candidates[0])
        out_matches[sample_row] = int(chunk_global_indices[chunk_row])
        method_out[sample_row] = "raw_xyz"


def pack_xy(ix: np.ndarray, iy: np.ndarray) -> np.ndarray:
    ix = np.asarray(ix, dtype=np.int64)
    iy = np.asarray(iy, dtype=np.int64)
    return ((ix & np.int64(0xffffffff)) << np.int64(32)) | (iy & np.int64(0xffffffff))


def unpack_xy(keys: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    keys = np.asarray(keys, dtype=np.uint64)
    ix = ((keys >> np.uint64(32)).astype(np.uint32)).view(np.int32).astype(np.int64)
    iy = (keys.astype(np.uint32)).view(np.int32).astype(np.int64)
    return ix, iy


def update_grid_stats_dict(state: Dict[int, List[float]], xyz: np.ndarray, grid_size: float) -> None:
    ix = np.floor(xyz[:, 0] / grid_size).astype(np.int64)
    iy = np.floor(xyz[:, 1] / grid_size).astype(np.int64)
    z = xyz[:, 2].astype(np.float64, copy=False)
    x = xyz[:, 0].astype(np.float64, copy=False)
    y = xyz[:, 1].astype(np.float64, copy=False)

    keys = pack_xy(ix, iy)
    uniq, inv = np.unique(keys, return_inverse=True)

    count = np.bincount(inv).astype(np.int64, copy=False)
    sumx = np.bincount(inv, weights=x)
    sumy = np.bincount(inv, weights=y)
    sumz = np.bincount(inv, weights=z)
    sumz2 = np.bincount(inv, weights=z * z)

    minz = np.full(len(uniq), np.inf, dtype=np.float64)
    maxz = np.full(len(uniq), -np.inf, dtype=np.float64)
    np.minimum.at(minz, inv, z)
    np.maximum.at(maxz, inv, z)

    for j, key in enumerate(uniq):
        k = int(key)
        cur = state.get(k)
        if cur is None:
            state[k] = [
                int(count[j]),
                float(sumx[j]),
                float(sumy[j]),
                float(sumz[j]),
                float(sumz2[j]),
                float(minz[j]),
                float(maxz[j]),
            ]
        else:
            cur[0] += int(count[j])
            cur[1] += float(sumx[j])
            cur[2] += float(sumy[j])
            cur[3] += float(sumz[j])
            cur[4] += float(sumz2[j])
            cur[5] = min(cur[5], float(minz[j]))
            cur[6] = max(cur[6], float(maxz[j]))


def finalize_grid_artifact(state: Dict[int, List[float]], grid_size: float) -> Dict[str, np.ndarray]:
    keys = np.array(sorted(state.keys()), dtype=np.int64)
    vals = np.array([state[int(k)] for k in keys], dtype=np.float64)

    count = vals[:, 0].astype(np.int32)
    sumx = vals[:, 1]
    sumy = vals[:, 2]
    sumz = vals[:, 3]
    sumz2 = vals[:, 4]
    minz = vals[:, 5].astype(np.float32)
    maxz = vals[:, 6].astype(np.float32)

    meanx = (sumx / np.maximum(count, 1)).astype(np.float32)
    meany = (sumy / np.maximum(count, 1)).astype(np.float32)
    meanz = (sumz / np.maximum(count, 1)).astype(np.float32)
    varz = np.maximum(sumz2 / np.maximum(count, 1) - (sumz / np.maximum(count, 1)) ** 2, 0.0)
    zstd = np.sqrt(varz).astype(np.float32)
    zrange = (maxz - minz).astype(np.float32)
    density = (count.astype(np.float32) / np.float32(grid_size * grid_size))

    return {
        "grid_size": np.float32(grid_size),
        "keys": keys,
        "count": count,
        "meanx": meanx,
        "meany": meany,
        "meanz": meanz,
        "minz": minz,
        "maxz": maxz,
        "zstd": zstd,
        "zrange": zrange,
        "density": density,
    }


def lookup_grid_rows(artifact: Dict[str, np.ndarray], ix: np.ndarray, iy: np.ndarray) -> np.ndarray:
    query = pack_xy(ix, iy)
    keys = artifact["keys"]
    pos = np.searchsorted(keys, query)
    out = np.full(len(query), -1, dtype=np.int64)
    valid = pos < len(keys)
    valid &= keys[np.minimum(pos, len(keys) - 1)] == query
    out[valid] = pos[valid]
    return out


def gather_with_default(arr: np.ndarray, rows: np.ndarray, default: float = 0.0, dtype=np.float32) -> np.ndarray:
    out = np.full(len(rows), default, dtype=dtype)
    good = rows >= 0
    if np.any(good):
        out[good] = arr[rows[good]]
    return out


def build_support_index(artifact: Dict[str, np.ndarray]) -> Dict[str, object]:
    coords = np.column_stack([artifact["meanx"], artifact["meany"], artifact["meanz"]]).astype(np.float32)
    if len(coords) > SUPPORT_MAX_CELLS:
        rng = np.random.default_rng(RANDOM_STATE)
        weights = artifact["count"].astype(np.float64)
        weights = weights / np.maximum(weights.sum(), 1.0)
        keep = rng.choice(len(coords), size=SUPPORT_MAX_CELLS, replace=False, p=weights)
        keep = np.sort(keep)
        reduced = {k: v[keep] if isinstance(v, np.ndarray) and len(v) == len(coords) else v for k, v in artifact.items()}
        reduced["keys"] = artifact["keys"][keep]
        coords = coords[keep]
        artifact = reduced
    xy_coords = coords[:, :2].astype(np.float32, copy=False)
    tree = cKDTree(coords)
    xy_tree = cKDTree(xy_coords)
    return {
        "artifact": artifact,
        "coords": coords,
        "xy_coords": xy_coords,
        "tree": tree,
        "xy_tree": xy_tree,
    }


def save_memmap_manifest(manifest: Dict[str, object], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)


def load_memmap_from_manifest(path: Path, spec: Dict[str, object]) -> np.memmap:
    return np.memmap(path, dtype=np.dtype(spec["dtype"]), mode="r+", shape=tuple(spec["shape"]))


def chunked_nearest_match_unresolved(
    sample_df: pd.DataFrame,
    out_matches: np.ndarray,
    method_out: np.ndarray,
    xyz_mm: np.memmap,
    tolerance: float = MATCH_NEAREST_TOLERANCE,
    chunk_points: int = CHUNK_READ_POINTS,
    sample_batch: int = MATCH_NEAREST_SAMPLE_BATCH,
    desc: str = "nearest rematch",
) -> int:
    unresolved = np.where(out_matches < 0)[0]
    if len(unresolved) == 0:
        return 0

    sample_xyz = sample_df.loc[unresolved, ["x", "y", "z"]].to_numpy(np.float32, copy=True)
    best_dist = np.full(len(unresolved), np.inf, dtype=np.float32)
    best_idx = np.full(len(unresolved), -1, dtype=np.int64)

    n_points = int(xyz_mm.shape[0])
    total_chunks = math.ceil(n_points / chunk_points)

    for start in tqdm(range(0, n_points, chunk_points), total=total_chunks, desc=desc):
        end = min(start + chunk_points, n_points)
        xyz_chunk = np.asarray(xyz_mm[start:end], dtype=np.float32)
        tree = cKDTree(xyz_chunk)

        for lo in range(0, len(sample_xyz), sample_batch):
            hi = min(lo + sample_batch, len(sample_xyz))
            d, idx = tree.query(sample_xyz[lo:hi], k=1, workers=-1)
            d = np.asarray(d, dtype=np.float32)
            idx = np.asarray(idx, dtype=np.int64)

            improve = d < best_dist[lo:hi]
            if np.any(improve):
                best_dist[lo:hi][improve] = d[improve]
                best_idx[lo:hi][improve] = start + idx[improve]

        del tree

    ok = np.isfinite(best_dist) & (best_dist <= np.float32(tolerance))
    if np.any(ok):
        target_rows = unresolved[ok]
        out_matches[target_rows] = best_idx[ok]
        method_out[target_rows] = "global_nearest_xyz"

    return int(np.sum(ok))


def require_nonempty_test_meta(test_meta: pd.DataFrame) -> None:
    if len(test_meta) == 0:
        raise ValueError(
            "No matched test rows remain after rebuilding matches. Delete the cache folder or set FORCE_REBUILD_ORIGINAL_CACHE = True once, then raise MATCH_NEAREST_TOLERANCE slightly if needed."
        )


def build_ground_proxy_artifact(
    grid_artifact: Dict[str, np.ndarray],
    opening_size: int = GROUND_PROXY_OPENING_SIZE,
    median_size: int = GROUND_PROXY_MEDIAN_SIZE,
    smooth_sigma: float = GROUND_PROXY_SMOOTH_SIGMA,
) -> Dict[str, object]:
    grid_size = float(grid_artifact["grid_size"])
    if not math.isclose(grid_size, float(GROUND_PROXY_GRID_SIZE), rel_tol=0.0, abs_tol=1e-9):
        raise ValueError(f"Ground proxy grid size mismatch: expected {GROUND_PROXY_GRID_SIZE}, got {grid_size}")

    ix, iy = unpack_xy(grid_artifact["keys"])
    min_ix = int(ix.min())
    max_ix = int(ix.max())
    min_iy = int(iy.min())
    max_iy = int(iy.max())

    width = int(max_ix - min_ix + 1)
    height = int(max_iy - min_iy + 1)

    observed_minz = np.full((height, width), np.nan, dtype=np.float32)
    observed_count = np.zeros((height, width), dtype=np.int32)

    col = (ix - min_ix).astype(np.int64, copy=False)
    row = (iy - min_iy).astype(np.int64, copy=False)
    observed_minz[row, col] = grid_artifact["minz"].astype(np.float32, copy=False)
    observed_count[row, col] = grid_artifact["count"].astype(np.int32, copy=False)

    valid = np.isfinite(observed_minz)
    if not np.any(valid):
        raise ValueError("Ground proxy build failed because no valid grid cells were available.")

    if np.all(valid):
        nearest_filled = observed_minz.copy()
    else:
        nearest_idx = ndimage.distance_transform_edt(~valid, return_distances=False, return_indices=True)
        nearest_filled = observed_minz[tuple(nearest_idx)].astype(np.float32, copy=False)

    opening_size = int(max(1, opening_size))
    if opening_size % 2 == 0:
        opening_size += 1

    median_size = int(max(1, median_size))
    if median_size % 2 == 0:
        median_size += 1

    ground = nearest_filled.astype(np.float32, copy=True)
    if opening_size > 1:
        ground = ndimage.grey_opening(ground, size=(opening_size, opening_size)).astype(np.float32, copy=False)
    if median_size > 1:
        ground = ndimage.median_filter(ground, size=median_size).astype(np.float32, copy=False)
    if float(smooth_sigma) > 0.0:
        ground = ndimage.gaussian_filter(ground, sigma=float(smooth_sigma)).astype(np.float32, copy=False)

    slope_y, slope_x = np.gradient(ground.astype(np.float32, copy=False), grid_size, grid_size)
    slope_mag = np.sqrt(slope_x * slope_x + slope_y * slope_y).astype(np.float32)
    curvature = ndimage.laplace(ground.astype(np.float32, copy=False)).astype(np.float32)

    return {
        "grid_size": np.float32(grid_size),
        "min_ix": min_ix,
        "min_iy": min_iy,
        "max_ix": max_ix,
        "max_iy": max_iy,
        "ground_z": ground.astype(np.float32, copy=False),
        "observed_minz": observed_minz.astype(np.float32, copy=False),
        "nearest_filled_minz": nearest_filled.astype(np.float32, copy=False),
        "count_grid": observed_count.astype(np.int32, copy=False),
        "valid_mask": valid.astype(bool, copy=False),
        "slope_x": slope_x.astype(np.float32, copy=False),
        "slope_y": slope_y.astype(np.float32, copy=False),
        "slope_mag": slope_mag.astype(np.float32, copy=False),
        "curvature": curvature.astype(np.float32, copy=False),
        "opening_size": int(opening_size),
        "median_size": int(median_size),
        "smooth_sigma": float(smooth_sigma),
    }


def get_or_build_ground_proxy_artifact(
    grid_artifact: Dict[str, np.ndarray],
    cache_path: Path,
    force: bool = False,
) -> Dict[str, object]:
    if cache_path.exists() and not force:
        cached = joblib.load(cache_path)
        if math.isclose(float(cached.get("grid_size", -1.0)), float(GROUND_PROXY_GRID_SIZE), rel_tol=0.0, abs_tol=1e-9):
            return cached

    artifact = build_ground_proxy_artifact(grid_artifact)
    joblib.dump(artifact, cache_path)
    return artifact


## Block summary: cache construction, multiscale grids, and matched metadata

**Methods used here**

- build memmapped arrays for the original cloud
- cache train/test matched metadata
- precompute multiscale grid artifacts and a support index
- keep the pipeline scalable without building a giant full-scene neighbor tree for every experiment

**Why this block exists**

This is a scalability block. It lets the notebook compute large-cloud context features without exploding memory. That design choice is also consistent with later urban LiDAR work that uses voxel or reduced primitives to keep large datasets tractable. [Aljumaily 2023 PCVC]


In [4]:
def prepare_original_cache_and_matches(force: bool = False):
    manifest_path = ARRAY_DIR / "manifest.json"
    train_meta_path = CACHE_DIR / "train_meta_matched.joblib"
    test_meta_path = CACHE_DIR / "test_meta_matched.joblib"
    grid_paths = {g: CACHE_DIR / f"grid_{str(g).replace('.', 'p')}.joblib" for g in GRID_SIZES}
    support_path = CACHE_DIR / "support_artifact.joblib"

    can_try_cache = (
        not force
        and manifest_path.exists()
        and train_meta_path.exists()
        and test_meta_path.exists()
        and support_path.exists()
        and all(path.exists() for path in grid_paths.values())
    )

    if can_try_cache:
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)

        train_meta = joblib.load(train_meta_path)
        test_meta = joblib.load(test_meta_path)

        cache_ok = (
            manifest.get("cache_version") == CACHE_VERSION
            and len(train_meta) > 0
            and len(test_meta) > 0
        )

        if cache_ok:
            xyz_mm = load_memmap_from_manifest(Path(manifest["xyz"]["path"]), manifest["xyz"])
            intrinsic_mms = {
                name: load_memmap_from_manifest(Path(spec["path"]), spec)
                for name, spec in manifest["intrinsics"].items()
            }
            grid_artifacts = {float(g): joblib.load(path) for g, path in grid_paths.items()}
            support_artifact = joblib.load(support_path)
            support_index = build_support_index(support_artifact)
            return manifest, xyz_mm, intrinsic_mms, train_meta, test_meta, grid_artifacts, support_index

        print("Cached match artifacts were stale or had an empty test split. Rebuilding cache.")

    original_path = find_original_las(ROOT)
    train_meta = stratified_cap_dataframe(load_sample_metadata(TRAIN_DIR), "label", MAX_TRAIN_SAMPLES)
    test_meta = stratified_cap_dataframe(load_sample_metadata(TEST_DIR), "label", MAX_TEST_SAMPLES)

    train_index = init_sorted_sample_key_index(train_meta)
    test_index = init_sorted_sample_key_index(test_meta)
    train_matches = np.full(len(train_meta), -1, dtype=np.int64)
    test_matches = np.full(len(test_meta), -1, dtype=np.int64)
    train_methods = np.full(len(train_meta), "", dtype=object)
    test_methods = np.full(len(test_meta), "", dtype=object)

    with laspy.open(original_path) as reader:
        header = reader.header
        n_points = int(header.point_count)
        scales = np.asarray(header.scales, dtype=np.float64)
        offsets = np.asarray(header.offsets, dtype=np.float64)
        effective_match_tolerance = max(float(np.max(scales)) * 4.0, float(MATCH_NEAREST_TOLERANCE))

        # Determine available dimensions from a probe chunk.
        first_chunk = next(reader.chunk_iterator(min(CHUNK_READ_POINTS, n_points)))
        dim_names = chunk_dimension_names(first_chunk)
        intrinsic_candidates = [
            "intensity",
            "return_number",
            "number_of_returns",
            "scan_angle",
            "scan_angle_rank",
            "red",
            "green",
            "blue",
        ]
        intrinsic_present = [name for name in intrinsic_candidates if get_dimension_name_from_names(dim_names, name) is not None]

    xyz_path = ARRAY_DIR / "xyz_float32.mmap"
    xyz_mm = np.memmap(xyz_path, dtype=np.float32, mode="w+", shape=(n_points, 3))
    intrinsic_mms = {}
    intrinsic_specs = {}
    for name in intrinsic_present:
        path = ARRAY_DIR / f"{name}.mmap"
        intrinsic_mms[name] = np.memmap(path, dtype=np.float32, mode="w+", shape=(n_points,))
        intrinsic_specs[name] = {
            "path": str(path),
            "dtype": "float32",
            "shape": [n_points],
        }

    grid_states = {float(g): {} for g in sorted(set(GRID_SIZES + [SUPPORT_GRID_SIZE]))}

    with laspy.open(original_path) as reader:
        start = 0
        total_chunks = math.ceil(n_points / CHUNK_READ_POINTS)
        for points in tqdm(reader.chunk_iterator(CHUNK_READ_POINTS), total=total_chunks, desc="scan original"):
            m = len(points)
            chunk_slice = slice(start, start + m)

            raw_xyz = get_raw_xyz_from_points(points)
            xyz = get_scaled_xyz_from_points(points, scales, offsets)
            xyz_mm[chunk_slice] = xyz

            ids = get_chunk_dim(points, "Id")
            if ids is None:
                chunk_ids = np.arange(start, start + m, dtype=np.int64)
            else:
                chunk_ids = np.asarray(ids).astype(np.int64, copy=False)

            assign_matches_from_chunk(
                np.arange(start, start + m, dtype=np.int64),
                raw_xyz,
                chunk_ids,
                train_index,
                train_matches,
                train_methods,
            )
            assign_matches_from_chunk(
                np.arange(start, start + m, dtype=np.int64),
                raw_xyz,
                chunk_ids,
                test_index,
                test_matches,
                test_methods,
            )

            for name in intrinsic_present:
                arr = get_chunk_dim(points, name)
                if arr is None:
                    intrinsic_mms[name][chunk_slice] = 0.0
                else:
                    intrinsic_mms[name][chunk_slice] = np.asarray(arr).astype(np.float32, copy=False)

            for g in grid_states:
                update_grid_stats_dict(grid_states[g], xyz, float(g))

            start += m

    xyz_mm.flush()
    for arr in intrinsic_mms.values():
        arr.flush()

    if MATCH_NEAREST_FALLBACK:
        matched_train = chunked_nearest_match_unresolved(
            train_meta,
            train_matches,
            train_methods,
            xyz_mm,
            tolerance=effective_match_tolerance,
            chunk_points=CHUNK_READ_POINTS,
            sample_batch=MATCH_NEAREST_SAMPLE_BATCH,
            desc="nearest rematch train",
        )
        matched_test = chunked_nearest_match_unresolved(
            test_meta,
            test_matches,
            test_methods,
            xyz_mm,
            tolerance=effective_match_tolerance,
            chunk_points=CHUNK_READ_POINTS,
            sample_batch=MATCH_NEAREST_SAMPLE_BATCH,
            desc="nearest rematch test",
        )
        print(f"Nearest fallback matched train rows: {matched_train:,}")
        print(f"Nearest fallback matched test rows: {matched_test:,}")

    train_meta = train_meta.copy()
    test_meta = test_meta.copy()
    train_meta["original_index"] = train_matches
    train_meta["match_method"] = train_methods
    test_meta["original_index"] = test_matches
    test_meta["match_method"] = test_methods

    train_meta = train_meta.loc[train_meta["original_index"] >= 0].reset_index(drop=True)
    test_meta = test_meta.loc[test_meta["original_index"] >= 0].reset_index(drop=True)

    grid_artifacts = {}
    for g, state in grid_states.items():
        artifact = finalize_grid_artifact(state, g)
        joblib.dump(artifact, grid_paths[float(g)])
        grid_artifacts[float(g)] = artifact

    support_artifact = grid_artifacts[float(SUPPORT_GRID_SIZE)]
    joblib.dump(support_artifact, support_path)
    support_index = build_support_index(support_artifact)

    manifest = {
        "original_las_path": str(original_path),
        "point_count": n_points,
        "xyz": {
            "path": str(xyz_path),
            "dtype": "float32",
            "shape": [n_points, 3],
        },
        "intrinsics": intrinsic_specs,
        "grid_sizes": GRID_SIZES,
        "support_grid_size": SUPPORT_GRID_SIZE,
        "class_names": CLASS_NAMES,
        "cache_version": CACHE_VERSION,
    }
    save_memmap_manifest(manifest, manifest_path)
    joblib.dump(train_meta, train_meta_path)
    joblib.dump(test_meta, test_meta_path)

    return manifest, xyz_mm, intrinsic_mms, train_meta, test_meta, grid_artifacts, support_index


MANIFEST, XYZ_MM, INTRINSIC_MMS, TRAIN_META, TEST_META, GRID_ARTIFACTS, SUPPORT_INDEX = prepare_original_cache_and_matches(force=FORCE_REBUILD_ORIGINAL_CACHE)
ORIGINAL_LAS_PATH = Path(MANIFEST["original_las_path"])

print("Original LAS:", ORIGINAL_LAS_PATH)
print("Original point count:", f"{int(MANIFEST['point_count']):,}")
print()
print("Matched train counts:")
display(TRAIN_META.groupby("class_name").size().rename("count").to_frame())
print()
print("Matched test counts:")
display(TEST_META.groupby("class_name").size().rename("count").to_frame())

require_nonempty_test_meta(TEST_META)




GROUND_PROXY_CACHE_PATH = CACHE_DIR / GROUND_PROXY_CACHE_NAME
GROUND_PROXY_ARTIFACT = get_or_build_ground_proxy_artifact(
    GRID_ARTIFACTS[float(GROUND_PROXY_GRID_SIZE)],
    GROUND_PROXY_CACHE_PATH,
    force=FORCE_REBUILD_ORIGINAL_CACHE,
)

print("Ground proxy artifact:")
print({
    "grid_size": float(GROUND_PROXY_ARTIFACT["grid_size"]),
    "opening_size": int(GROUND_PROXY_ARTIFACT["opening_size"]),
    "median_size": int(GROUND_PROXY_ARTIFACT["median_size"]),
    "smooth_sigma": float(GROUND_PROXY_ARTIFACT["smooth_sigma"]),
    "grid_shape": tuple(int(v) for v in GROUND_PROXY_ARTIFACT["ground_z"].shape),
})


Original LAS: original_point_cloud.las
Original point count: 18,854,300

Matched train counts:


,count
class_name,
Buildings,24521
Cars,1928
Ground,81746
Trees,40642



Matched test counts:


,count
class_name,
Buildings,33023
Cars,780
Ground,111130
Trees,31598


Ground proxy artifact:
{'grid_size': 1.0, 'opening_size': 5, 'median_size': 3, 'smooth_sigma': 1.0, 'grid_shape': (5100, 4426)}


## Block summary: engineered feature families

**Methods used here**

- multiscale cell and neighborhood statistics
- a grid-derived **ground proxy surface** for a more explicit terrain normalization signal
- broader XY terrain proxies for additional height-above-ground stabilization
- local covariance geometry features such as linearity, planarity, sphericity, anisotropy, omnivariance, surface variation, normal-z, and verticality
- return, intensity, and RGB-derived features when available

**Why this block exists**

These are the classical feature families most likely to move the current failure modes:

- better normalized-height features help separate flat roofs from true ground
- eigenvalue-derived local geometry helps separate planar roofs from irregular crowns
- multi-scale neighborhoods improve robustness because different structures become distinctive at different support sizes
- radiometric and return cues can help disambiguate vegetation and built surfaces when the scanner recorded them


In [5]:

def safe_div(a, b, default=0.0):
    a_arr, b_arr = np.broadcast_arrays(
        np.asarray(a, dtype=np.float32),
        np.asarray(b, dtype=np.float32),
    )
    out = np.full(a_arr.shape, np.float32(default), dtype=np.float32)
    mask = np.abs(b_arr) > np.float32(1e-12)
    if np.any(mask):
        out[mask] = (a_arr[mask] / b_arr[mask]).astype(np.float32, copy=False)
    return out


def _safe_tree_query(tree: cKDTree, query_points: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    k = int(max(1, min(int(k), int(tree.n))))
    dists, idx = tree.query(query_points, k=k, workers=-1)
    if k == 1:
        dists = dists[:, None]
        idx = idx[:, None]
    return np.asarray(dists, dtype=np.float32), np.asarray(idx, dtype=np.int64)


def _weighted_local_geometry_features(
    neigh_xyz: np.ndarray,
    neigh_weights: np.ndarray,
    eps: float = 1e-9,
) -> Dict[str, np.ndarray]:
    w = np.asarray(neigh_weights, dtype=np.float32)
    xyz = np.asarray(neigh_xyz, dtype=np.float32)

    w = np.maximum(w, np.float32(eps))
    wsum = np.maximum(w.sum(axis=1, keepdims=True), np.float32(eps))
    center = (xyz * w[..., None]).sum(axis=1, keepdims=True) / wsum[..., None]
    centered = xyz - center

    cov = np.einsum("nki,nkj,nk->nij", centered, centered, w, optimize=True)
    cov = cov / wsum[:, :, None]
    cov = 0.5 * (cov + np.transpose(cov, (0, 2, 1)))

    eigvals, eigvecs = np.linalg.eigh(cov)
    eigvals = np.clip(eigvals.astype(np.float32, copy=False), np.float32(eps), None)

    l3 = eigvals[:, 0]
    l2 = eigvals[:, 1]
    l1 = eigvals[:, 2]
    lsum = np.maximum(l1 + l2 + l3, np.float32(eps))

    p1 = l1 / lsum
    p2 = l2 / lsum
    p3 = l3 / lsum

    normal_vec = eigvecs[:, :, 0].astype(np.float32, copy=False)
    normal_z_abs = np.abs(normal_vec[:, 2])

    zvals = xyz[:, :, 2]
    z_center = center[:, 0, 2]

    return {
        "linearity": safe_div(l1 - l2, l1, default=0.0),
        "planarity": safe_div(l2 - l3, l1, default=0.0),
        "sphericity": safe_div(l3, l1, default=0.0),
        "anisotropy": safe_div(l1 - l3, l1, default=0.0),
        "omnivariance": np.cbrt(np.maximum(l1 * l2 * l3, np.float32(0.0))).astype(np.float32),
        "eigenentropy": (-(p1 * np.log(p1 + eps) + p2 * np.log(p2 + eps) + p3 * np.log(p3 + eps))).astype(np.float32),
        "surface_variation": safe_div(l3, lsum, default=0.0),
        "normal_z_abs": normal_z_abs.astype(np.float32, copy=False),
        "verticality": (1.0 - normal_z_abs).astype(np.float32, copy=False),
        "z_std": zvals.std(axis=1).astype(np.float32),
        "z_range": (zvals.max(axis=1) - zvals.min(axis=1)).astype(np.float32),
        "z_center_offset": (zvals.mean(axis=1) - z_center).astype(np.float32),
    }


def build_grid_features_for_batch(
    xyz_batch: np.ndarray,
    grid_artifact: Dict[str, np.ndarray],
    requested_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    req = None if requested_columns is None else set(requested_columns)
    g = float(grid_artifact["grid_size"])
    suffix = f"_g{g}"

    ix = np.floor(xyz_batch[:, 0] / g).astype(np.int64)
    iy = np.floor(xyz_batch[:, 1] / g).astype(np.int64)
    rows = lookup_grid_rows(grid_artifact, ix, iy)

    count = gather_with_default(grid_artifact["count"], rows, 0.0, np.float32)
    density = gather_with_default(grid_artifact["density"], rows, 0.0, np.float32)
    meanx = gather_with_default(grid_artifact["meanx"], rows, 0.0, np.float32)
    meany = gather_with_default(grid_artifact["meany"], rows, 0.0, np.float32)
    meanz = gather_with_default(grid_artifact["meanz"], rows, 0.0, np.float32)
    minz = gather_with_default(grid_artifact["minz"], rows, 0.0, np.float32)
    zstd = gather_with_default(grid_artifact["zstd"], rows, 0.0, np.float32)
    zrange = gather_with_default(grid_artifact["zrange"], rows, 0.0, np.float32)

    feats = {}
    raw = {
        f"cell_count{suffix}": count,
        f"cell_density{suffix}": density,
        f"cell_zstd{suffix}": zstd,
        f"cell_zrange{suffix}": zrange,
        f"hag_min{suffix}": xyz_batch[:, 2] - minz,
        f"hag_mean{suffix}": xyz_batch[:, 2] - meanz,
        f"cell_xoffset{suffix}": xyz_batch[:, 0] - meanx,
        f"cell_yoffset{suffix}": xyz_batch[:, 1] - meany,
    }
    for name, values in raw.items():
        if req is None or name in req:
            feats[name] = values.astype(np.float32, copy=False)

    left_rows = lookup_grid_rows(grid_artifact, ix - 1, iy)
    right_rows = lookup_grid_rows(grid_artifact, ix + 1, iy)
    down_rows = lookup_grid_rows(grid_artifact, ix, iy - 1)
    up_rows = lookup_grid_rows(grid_artifact, ix, iy + 1)

    default_min = float(np.nanmean(minz)) if len(minz) else 0.0
    left_min = gather_with_default(grid_artifact["minz"], left_rows, default_min, np.float32)
    right_min = gather_with_default(grid_artifact["minz"], right_rows, default_min, np.float32)
    down_min = gather_with_default(grid_artifact["minz"], down_rows, default_min, np.float32)
    up_min = gather_with_default(grid_artifact["minz"], up_rows, default_min, np.float32)

    slope_x = (right_min - left_min) / np.float32(max(2.0 * g, 1e-6))
    slope_y = (up_min - down_min) / np.float32(max(2.0 * g, 1e-6))
    slope_mag = np.sqrt(slope_x * slope_x + slope_y * slope_y).astype(np.float32)

    if req is None or f"ground_slope_mag{suffix}" in req:
        feats[f"ground_slope_mag{suffix}"] = slope_mag

    count_stack = []
    minz_stack = []
    zrange_stack = []
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            nbr_rows = lookup_grid_rows(grid_artifact, ix + dx, iy + dy)
            count_stack.append(gather_with_default(grid_artifact["count"], nbr_rows, 0.0, np.float32))
            minz_stack.append(gather_with_default(grid_artifact["minz"], nbr_rows, np.nan, np.float32))
            zrange_stack.append(gather_with_default(grid_artifact["zrange"], nbr_rows, 0.0, np.float32))

    count_stack_arr = np.stack(count_stack, axis=1)
    minz_stack_arr = np.stack(minz_stack, axis=1)
    zrange_stack_arr = np.stack(zrange_stack, axis=1)

    pooled = {
        f"nbr_count_sum{suffix}": np.nansum(count_stack_arr, axis=1).astype(np.float32),
        f"nbr_count_mean{suffix}": np.nanmean(count_stack_arr, axis=1).astype(np.float32),
        f"nbr_ground_relief{suffix}": np.nan_to_num(np.nanmax(minz_stack_arr, axis=1) - np.nanmin(minz_stack_arr, axis=1), nan=0.0).astype(np.float32),
        f"nbr_zrange_mean{suffix}": np.nanmean(zrange_stack_arr, axis=1).astype(np.float32),
    }
    for name, values in pooled.items():
        if req is None or name in req:
            feats[name] = values.astype(np.float32, copy=False)

    return pd.DataFrame(feats)


def build_support_features_for_batch(
    xyz_batch: np.ndarray,
    support_index: Dict[str, object],
    requested_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    req = None if requested_columns is None else set(requested_columns)
    tree_3d: cKDTree = support_index["tree"]
    tree_xy: cKDTree = support_index["xy_tree"]
    art = support_index["artifact"]
    coords = support_index["coords"]

    max_k_3d = int(max(SUPPORT_KS + SUPPORT_GEOM_KS + SUPPORT_ADAPTIVE_GEOM_KS))
    dists_3d, idx_3d = _safe_tree_query(tree_3d, xyz_batch, max_k_3d)

    feats = {}
    for k in SUPPORT_KS:
        dk = dists_3d[:, :k]
        ik = idx_3d[:, :k]
        count_vals = art["count"][ik].astype(np.float32)
        zstd_vals = art["zstd"][ik].astype(np.float32)
        zrange_vals = art["zrange"][ik].astype(np.float32)
        density_vals = art["density"][ik].astype(np.float32)

        block = {
            f"support_mean_d_k{k}": dk.mean(axis=1).astype(np.float32),
            f"support_std_d_k{k}": dk.std(axis=1).astype(np.float32),
            f"support_max_d_k{k}": dk.max(axis=1).astype(np.float32),
            f"support_density_proxy_k{k}": safe_div(float(k), dk[:, -1] ** 3 + 1e-6, default=0.0),
            f"support_mean_count_k{k}": count_vals.mean(axis=1).astype(np.float32),
            f"support_mean_density_k{k}": density_vals.mean(axis=1).astype(np.float32),
            f"support_mean_zstd_k{k}": zstd_vals.mean(axis=1).astype(np.float32),
            f"support_mean_zrange_k{k}": zrange_vals.mean(axis=1).astype(np.float32),
        }
        for name, values in block.items():
            if req is None or name in req:
                feats[name] = values.astype(np.float32, copy=False)

    for k in SUPPORT_GEOM_KS:
        ik = idx_3d[:, :k]
        neigh_xyz = coords[ik].astype(np.float32, copy=False)
        neigh_w = art["count"][ik].astype(np.float32, copy=False)
        geom = _weighted_local_geometry_features(neigh_xyz, neigh_w)

        block = {
            f"geom_linearity_k{k}": geom["linearity"],
            f"geom_planarity_k{k}": geom["planarity"],
            f"geom_sphericity_k{k}": geom["sphericity"],
            f"geom_anisotropy_k{k}": geom["anisotropy"],
            f"geom_omnivariance_k{k}": geom["omnivariance"],
            f"geom_eigenentropy_k{k}": geom["eigenentropy"],
            f"geom_surface_variation_k{k}": geom["surface_variation"],
            f"geom_normal_z_abs_k{k}": geom["normal_z_abs"],
            f"geom_verticality_k{k}": geom["verticality"],
            f"geom_z_std_k{k}": geom["z_std"],
            f"geom_z_range_k{k}": geom["z_range"],
        }
        for name, values in block.items():
            if req is None or name in req:
                feats[name] = values.astype(np.float32, copy=False)

    adapt_geom_candidates = {}
    adapt_omni = []
    for k in SUPPORT_ADAPTIVE_GEOM_KS:
        ik = idx_3d[:, :k]
        neigh_xyz = coords[ik].astype(np.float32, copy=False)
        neigh_w = art["count"][ik].astype(np.float32, copy=False)
        geom = _weighted_local_geometry_features(neigh_xyz, neigh_w)
        adapt_geom_candidates[int(k)] = geom
        adapt_omni.append(geom["omnivariance"].astype(np.float32, copy=False))

    adapt_omni = np.stack(adapt_omni, axis=1)
    adapt_choice = np.argmax(adapt_omni, axis=1).astype(np.int32)
    adapt_best_k = np.asarray([SUPPORT_ADAPTIVE_GEOM_KS[i] for i in adapt_choice], dtype=np.float32)

    def _pick_adapt_geom(key: str) -> np.ndarray:
        stacked = np.stack(
            [adapt_geom_candidates[int(k)][key].astype(np.float32, copy=False) for k in SUPPORT_ADAPTIVE_GEOM_KS],
            axis=1,
        )
        return stacked[np.arange(len(stacked)), adapt_choice].astype(np.float32, copy=False)

    adaptive_block = {
        "geom_adapt_best_k": adapt_best_k,
        "geom_adapt_linearity": _pick_adapt_geom("linearity"),
        "geom_adapt_planarity": _pick_adapt_geom("planarity"),
        "geom_adapt_sphericity": _pick_adapt_geom("sphericity"),
        "geom_adapt_anisotropy": _pick_adapt_geom("anisotropy"),
        "geom_adapt_omnivariance": _pick_adapt_geom("omnivariance"),
        "geom_adapt_eigenentropy": _pick_adapt_geom("eigenentropy"),
        "geom_adapt_surface_variation": _pick_adapt_geom("surface_variation"),
        "geom_adapt_normal_z_abs": _pick_adapt_geom("normal_z_abs"),
        "geom_adapt_verticality": _pick_adapt_geom("verticality"),
        "geom_adapt_z_std": _pick_adapt_geom("z_std"),
        "geom_adapt_z_range": _pick_adapt_geom("z_range"),
    }
    for name, values in adaptive_block.items():
        if req is None or name in req:
            feats[name] = values.astype(np.float32, copy=False)

    max_k_xy = int(max(SUPPORT_TERRAIN_KS))
    xy_query = np.asarray(xyz_batch[:, :2], dtype=np.float32)
    dists_xy, idx_xy = _safe_tree_query(tree_xy, xy_query, max_k_xy)

    for k in SUPPORT_TERRAIN_KS:
        ik = idx_xy[:, :k]
        terrain_minz = art["minz"][ik].astype(np.float32, copy=False)
        terrain_counts = art["count"][ik].astype(np.float32, copy=False)

        low_rank = max(0, int(math.ceil(0.10 * k)) - 1)
        q10 = np.partition(terrain_minz, low_rank, axis=1)[:, low_rank].astype(np.float32)
        tmin = terrain_minz.min(axis=1).astype(np.float32)
        tmean = terrain_minz.mean(axis=1).astype(np.float32)
        trelief = (terrain_minz.max(axis=1) - terrain_minz.min(axis=1)).astype(np.float32)
        dxy_mean = dists_xy[:, :k].mean(axis=1).astype(np.float32)

        block = {
            f"terrain_xy_min_k{k}": tmin,
            f"terrain_xy_q10_k{k}": q10,
            f"terrain_xy_mean_k{k}": tmean,
            f"terrain_xy_relief_k{k}": trelief,
            f"terrain_xy_neighbor_mean_d_k{k}": dxy_mean,
            f"terrain_xy_neighbor_mean_count_k{k}": terrain_counts.mean(axis=1).astype(np.float32),
            f"hag_terrain_xy_min_k{k}": (xyz_batch[:, 2] - tmin).astype(np.float32),
            f"hag_terrain_xy_q10_k{k}": (xyz_batch[:, 2] - q10).astype(np.float32),
            f"hag_terrain_xy_mean_k{k}": (xyz_batch[:, 2] - tmean).astype(np.float32),
        }
        for name, values in block.items():
            if req is None or name in req:
                feats[name] = values.astype(np.float32, copy=False)

    return pd.DataFrame(feats)


def build_ground_proxy_features_for_batch(
    xyz_batch: np.ndarray,
    ground_artifact: Dict[str, object],
    requested_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    req = None if requested_columns is None else set(requested_columns)
    g = float(ground_artifact["grid_size"])

    ix = np.floor(xyz_batch[:, 0] / g).astype(np.int64) - int(ground_artifact["min_ix"])
    iy = np.floor(xyz_batch[:, 1] / g).astype(np.int64) - int(ground_artifact["min_iy"])

    ix = np.clip(ix, 0, ground_artifact["ground_z"].shape[1] - 1)
    iy = np.clip(iy, 0, ground_artifact["ground_z"].shape[0] - 1)

    ground_z = ground_artifact["ground_z"][iy, ix].astype(np.float32, copy=False)
    observed_minz = ground_artifact["nearest_filled_minz"][iy, ix].astype(np.float32, copy=False)
    slope_x = ground_artifact["slope_x"][iy, ix].astype(np.float32, copy=False)
    slope_y = ground_artifact["slope_y"][iy, ix].astype(np.float32, copy=False)
    slope_mag = ground_artifact["slope_mag"][iy, ix].astype(np.float32, copy=False)
    curvature = ground_artifact["curvature"][iy, ix].astype(np.float32, copy=False)

    feats = {}
    raw = {
        "ground_proxy_z": ground_z,
        "hag_ground_proxy": (xyz_batch[:, 2] - ground_z).astype(np.float32),
        "ground_proxy_observed_minz": observed_minz,
        "ground_proxy_residual": (observed_minz - ground_z).astype(np.float32),
        "ground_proxy_slope_x": slope_x,
        "ground_proxy_slope_y": slope_y,
        "ground_proxy_slope_mag": slope_mag,
        "ground_proxy_curvature": curvature,
    }
    for name, values in raw.items():
        if req is None or name in req:
            feats[name] = values.astype(np.float32, copy=False)

    return pd.DataFrame(feats)


def build_intrinsic_features_for_batch(
    indices: np.ndarray,
    xyz_batch: np.ndarray,
    intrinsic_mms: Dict[str, np.memmap],
    requested_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    req = None if requested_columns is None else set(requested_columns)
    feats = {}

    base = {
        "x": xyz_batch[:, 0].astype(np.float32),
        "y": xyz_batch[:, 1].astype(np.float32),
        "z": xyz_batch[:, 2].astype(np.float32),
    }
    for name, values in base.items():
        if req is None or name in req:
            feats[name] = values

    raw_intrinsics = {}
    for name, arr in intrinsic_mms.items():
        raw_vals = np.asarray(arr[indices]).astype(np.float32, copy=False)
        raw_intrinsics[name] = raw_vals
        if req is None or name in req:
            feats[name] = raw_vals

    intensity = raw_intrinsics.get("intensity")
    if intensity is not None:
        derived = {
            "log_intensity": np.log1p(np.maximum(intensity, 0.0)).astype(np.float32),
        }
        for name, values in derived.items():
            if req is None or name in req:
                feats[name] = values

    scan_angle = raw_intrinsics.get("scan_angle")
    if scan_angle is None:
        scan_angle = raw_intrinsics.get("scan_angle_rank")
    if scan_angle is not None:
        derived = {
            "abs_scan_angle": np.abs(scan_angle).astype(np.float32),
        }
        for name, values in derived.items():
            if req is None or name in req:
                feats[name] = values

    return_number = raw_intrinsics.get("return_number")
    number_of_returns = raw_intrinsics.get("number_of_returns")
    if return_number is not None and number_of_returns is not None:
        denom = np.maximum(number_of_returns, 1.0).astype(np.float32)
        derived = {
            "return_fraction": safe_div(return_number, denom, default=0.0),
            "is_first_return": (return_number <= 1.0).astype(np.float32),
            "is_last_return": (return_number >= denom).astype(np.float32),
            "is_multi_return": (denom > 1.0).astype(np.float32),
        }
        for name, values in derived.items():
            if req is None or name in req:
                feats[name] = values.astype(np.float32, copy=False)

    red = raw_intrinsics.get("red")
    green = raw_intrinsics.get("green")
    blue = raw_intrinsics.get("blue")
    if red is not None and green is not None and blue is not None:
        rgb_sum = np.maximum(red + green + blue, 1.0).astype(np.float32)
        derived = {
            "rgb_brightness": (rgb_sum / 3.0).astype(np.float32),
            "rgb_r_norm": safe_div(red, rgb_sum, default=0.0),
            "rgb_g_norm": safe_div(green, rgb_sum, default=0.0),
            "rgb_b_norm": safe_div(blue, rgb_sum, default=0.0),
            "rgb_excess_green": (2.0 * green - red - blue).astype(np.float32),
        }
        for name, values in derived.items():
            if req is None or name in req:
                feats[name] = values.astype(np.float32, copy=False)

    return pd.DataFrame(feats)


def build_feature_table_for_indices(
    indices: np.ndarray,
    requested_columns: Optional[Sequence[str]] = None,
    verbose: bool = False,
) -> pd.DataFrame:
    req = None if requested_columns is None else set(requested_columns)
    indices = np.asarray(indices, dtype=np.int64)
    xyz_batch = np.asarray(XYZ_MM[indices], dtype=np.float32)

    blocks = []
    blocks.append(build_intrinsic_features_for_batch(indices, xyz_batch, INTRINSIC_MMS, requested_columns=requested_columns))

    for g in GRID_SIZES:
        blocks.append(build_grid_features_for_batch(xyz_batch, GRID_ARTIFACTS[float(g)], requested_columns=requested_columns))

    blocks.append(build_ground_proxy_features_for_batch(xyz_batch, GROUND_PROXY_ARTIFACT, requested_columns=requested_columns))
    blocks.append(build_support_features_for_batch(xyz_batch, SUPPORT_INDEX, requested_columns=requested_columns))

    df = pd.concat(blocks, axis=1)

    cross_features = {}
    if "hag_ground_proxy" in df.columns and "hag_terrain_xy_q10_k32" in df.columns:
        cross_features["hag_ground_proxy_minus_terrain_q10_k32"] = (
            df["hag_ground_proxy"].to_numpy(dtype=np.float32, copy=False)
            - df["hag_terrain_xy_q10_k32"].to_numpy(dtype=np.float32, copy=False)
        ).astype(np.float32, copy=False)

    if "hag_ground_proxy" in df.columns and "hag_min_g1.0" in df.columns:
        cross_features["hag_ground_proxy_minus_cell_min_g1.0"] = (
            df["hag_ground_proxy"].to_numpy(dtype=np.float32, copy=False)
            - df["hag_min_g1.0"].to_numpy(dtype=np.float32, copy=False)
        ).astype(np.float32, copy=False)

    if "ground_proxy_slope_mag" in df.columns and "geom_adapt_planarity" in df.columns:
        cross_features["ground_proxy_slope_planarity_ratio"] = safe_div(
            df["ground_proxy_slope_mag"].to_numpy(dtype=np.float32, copy=False),
            np.maximum(df["geom_adapt_planarity"].to_numpy(dtype=np.float32, copy=False), 1e-4),
            default=0.0,
        )

    for name, values in cross_features.items():
        if req is None or name in req:
            df[name] = values.astype(np.float32, copy=False)

    if requested_columns is not None:
        for col in requested_columns:
            if col not in df.columns:
                df[col] = np.nan
        df = df.loc[:, list(requested_columns)]

    df.insert(0, "original_index", indices)
    return df


def get_or_build_feature_table(
    sample_meta: pd.DataFrame,
    cache_prefix: str,
    force: bool = False,
    requested_columns: Optional[Sequence[str]] = None,
) -> pd.DataFrame:
    if requested_columns is None:
        cache_path = CACHE_DIR / f"{cache_prefix}_features_{FEATURE_CACHE_TAG}.joblib"
        if cache_path.exists() and not force:
            return joblib.load(cache_path)

    table = build_feature_table_for_indices(
        sample_meta["original_index"].to_numpy(np.int64),
        requested_columns=requested_columns,
        verbose=True,
    )
    if requested_columns is None:
        joblib.dump(table, cache_path)
    return table


## Block summary: sample feature table assembly

**Methods used here**

- build train and test feature tables from the original matched points
- optionally drop extreme outliers using a training-derived threshold
- keep the train and test feature spaces identical

**Why this block exists**

This is where the abstract feature ideas above become actual model matrices. The outlier gate is intentionally conservative because severe support-neighborhood failures can otherwise dominate the ranking stage.


In [6]:
TRAIN_FEATURES = get_or_build_feature_table(TRAIN_META, "train", force=FORCE_REBUILD_FEATURES)
TEST_FEATURES = get_or_build_feature_table(TEST_META, "test", force=FORCE_REBUILD_FEATURES)

ALL_FEATURE_NAMES = [c for c in TRAIN_FEATURES.columns if c != "original_index"]

X_train_full = TRAIN_FEATURES[ALL_FEATURE_NAMES].copy()
X_test_full = TEST_FEATURES[ALL_FEATURE_NAMES].copy()
y_train = TRAIN_META["label"].to_numpy(np.int32)
y_test = TEST_META["label"].to_numpy(np.int32)
target_names = [LABEL_TO_CLASS[i] for i in np.unique(y_train)]

if DROP_TRAIN_OUTLIERS and TRAIN_OUTLIER_FEATURE in X_train_full.columns:
    thresh = float(np.nanquantile(X_train_full[TRAIN_OUTLIER_FEATURE], TRAIN_OUTLIER_Q))
    train_vals = X_train_full[TRAIN_OUTLIER_FEATURE].to_numpy(np.float32)
    keep_train = np.isfinite(train_vals) & (train_vals <= thresh)

    if DROP_TEST_OUTLIERS_WITH_TRAIN_THRESHOLD and TRAIN_OUTLIER_FEATURE in X_test_full.columns and len(X_test_full) > 0:
        test_vals = X_test_full[TRAIN_OUTLIER_FEATURE].to_numpy(np.float32)
        keep_test = np.isfinite(test_vals) & (test_vals <= thresh)
    else:
        keep_test = np.ones(len(X_test_full), dtype=bool)

    TRAIN_META = TRAIN_META.loc[keep_train].reset_index(drop=True)
    TEST_META = TEST_META.loc[keep_test].reset_index(drop=True)
    X_train_full = X_train_full.loc[keep_train].reset_index(drop=True)
    X_test_full = X_test_full.loc[keep_test].reset_index(drop=True)
    y_train = y_train[keep_train]
    y_test = y_test[keep_test]

print("Feature count:", X_train_full.shape[1])
print("Train rows:", len(X_train_full))
print("Test rows:", len(X_test_full))

if len(X_train_full) == 0:
    raise ValueError("No training rows remain after matching/filtering. Check the sample-to-original matching and outlier settings.")

if len(X_test_full) == 0:
    msg = "Test feature table is empty after matching/filtering. Training will continue, but holdout evaluation will be skipped."
    if ALLOW_EMPTY_TEST_SPLIT:
        print("Warning:", msg)
    else:
        raise ValueError(msg)




Feature count: 174
Train rows: 148092
Test rows: 176531


## Block summary: filtering, ranking, and choosing the final feature set

This section reduces a large handcrafted feature table to a smaller set that is easier to train, easier to inspect, and less repetitive. The overall idea follows a common point-cloud workflow: remove weak or redundant variables, rank the survivors with more than one method, then keep a compact subset that still preserves the main geometric and terrain cues needed for class separation. Studies in 3D point-cloud classification have shown that both neighborhood design and feature relevance matter strongly for final classification quality, and that a smaller, more informative feature set can improve efficiency without giving up useful structure ([Weinmann 2014/2015 overview](https://pdfs.semanticscholar.org/d859/8a74f1dba12fd973f6ef1372e616f5f9c68a.pdf)).

### What this block does, in order

**1. Remove features with almost no variation**

The first pass drops columns whose values are nearly constant across the training set. A feature that barely changes usually contributes little to class separation, because it does not give the model meaningful contrast from one point to another. This is a simple screening step to avoid carrying dead columns farther into the pipeline. The code uses a very small variance threshold, so only features that are effectively flat are removed.

**2. Remove features that repeat the same information**

The next pass computes pairwise correlation and removes one feature from any pair that is almost identical to the other. In this notebook, that cutoff is set to `0.98`, which is a practical rule for removing near-duplicates rather than a universal standard. The goal is not to eliminate all related variables, but to reduce cases where many columns are describing almost the same pattern. That keeps the feature bank more compact and easier to interpret.

**3. Fill missing values with the column median**

After filtering, missing numeric values are filled with the median of each column. The scikit-learn [`SimpleImputer`](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) documentation describes median imputation as a standard numeric imputation strategy, and it is commonly used because it is less sensitive than the mean to unusually large or small values. That makes it a sensible default for mixed point-cloud features such as heights, local geometry summaries, and return-based attributes.

**4. Rank the remaining features from several different viewpoints**

The code does not rely on one ranking method alone. Instead, it estimates feature usefulness in several ways and then combines them.

- **Mutual information** measures how informative each feature is about the class label on its own. Scikit-learn provides [`mutual_info_classif`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html) specifically for univariate feature selection in classification problems.
- **ExtraTrees importance** measures how often and how effectively a feature helps split the data inside a randomized tree ensemble. Scikit-learn describes [`ExtraTreesClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html) as an ensemble of randomized decision trees whose predictions are combined to improve predictive performance.
- **Optional XGBoost importance** provides a second tree-based view using boosted trees rather than randomized bagged trees. XGBoost exposes feature importance through its Python API ([XGBoost Python API](https://xgboost.readthedocs.io/en/latest/python/python_api.html)).
- **Optional permutation importance** checks how much model performance drops when one feature is randomly shuffled. Scikit-learn defines permutation importance exactly this way in its user guide ([Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)).

Using several ranking methods helps reduce dependence on any one scoring rule. In practice, some features look good in a one-variable test, some matter mainly through tree splits, and some only show their value when tested by shuffling.

**5. Combine the rankings into one consensus score**

Each ranking list is converted into a weighted score, and the scores are summed into a single consensus ranking. Features that are supported by several ranking methods rise to the top. This favors variables that are useful in more than one sense instead of rewarding only one particular model family.

**6. Force-retain a small set of physically meaningful features**

After scoring, the notebook still force-keeps a short list of features that represent core spatial cues:

- point coordinates
- height-above-ground style terrain references
- local geometric shape descriptors such as planarity, sphericity, surface variation, anisotropy, verticality, and normal direction
- return-order features
- RGB color channels when available
- adaptive-neighborhood geometry summaries
- ground-proxy features added in the later keep-list

This step is important because purely score-driven selection can sometimes drop features that are physically meaningful but underrepresented in one training sample. In point-cloud classification, geometric descriptors and neighborhood-dependent summaries are known to be central to class separation, especially for distinguishing surfaces such as roofs, ground, and vegetation ([Weinmann review](https://pdfs.semanticscholar.org/d859/8a74f1dba12fd973f6ef1372e616f5f9c68a.pdf)).

**7. Build the final training and test tables from the selected columns**

The output of this block is a reduced feature table for model fitting, along with a saved ranking report. The report records the consensus score, the contribution from each ranking method, and whether a feature was retained because it belonged to the mandatory keep-list.

### Why these choices make sense for this notebook

This notebook starts from a broad feature bank on purpose, because point-cloud classes such as ground, buildings, trees, and cars can differ in several ways at once: elevation above local ground, local surface shape, return pattern, and sometimes color. But a large feature table quickly becomes repetitive. The workflow here is therefore a compromise:

- keep the feature bank broad enough to capture real scene structure
- remove dead and near-duplicate columns
- rank the survivors with several complementary methods
- protect a small set of domain-meaningful geometry and terrain variables from being discarded too early

That makes the final subset smaller and more stable while still preserving the cues that matter most for interpreting 3D urban structure.

In [7]:
def variance_filter(df: pd.DataFrame, threshold: float = 1e-10) -> List[str]:
    keep = []
    for c in df.columns:
        if np.nanvar(df[c].to_numpy(dtype=np.float64)) > threshold:
            keep.append(c)
    return keep


def correlation_prune(df: pd.DataFrame, columns: Sequence[str], threshold: float = 0.98) -> List[str]:
    corr = df[list(columns)].corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop = [col for col in upper.columns if any(upper[col] > threshold)]
    return [c for c in columns if c not in drop]


def rank_series_to_score(cols: Sequence[str], ranked: Sequence[str], top_k: int, weight: float = 1.0) -> pd.Series:
    ranked = list(ranked)[:top_k]
    vals = {c: 0.0 for c in cols}
    n = len(ranked)
    for i, c in enumerate(ranked):
        vals[c] = weight * (n - i) / max(n, 1)
    return pd.Series(vals)


def stratified_cap_rows(labels: np.ndarray, max_rows: int) -> np.ndarray:
    if max_rows is None or len(labels) <= max_rows:
        return np.arange(len(labels), dtype=np.int64)

    df = pd.DataFrame({"row_id": np.arange(len(labels), dtype=np.int64), "label": labels})
    capped = stratified_cap_dataframe(df, "label", max_rows, random_state=RANDOM_STATE)
    return capped["row_id"].to_numpy(np.int64)


base_keep = variance_filter(X_train_full)
X_train_v = X_train_full[base_keep].copy()
X_test_v = X_test_full[base_keep].copy()

corr_keep = correlation_prune(X_train_v, X_train_v.columns, threshold=0.98)
if len(corr_keep) > MAX_FEATURES_AFTER_CORR:
    corr_keep = corr_keep[:MAX_FEATURES_AFTER_CORR]

X_train_c = X_train_v[corr_keep].copy()
X_test_c = X_test_v[corr_keep].copy()

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_c), columns=X_train_c.columns)
if len(X_test_c) > 0:
    X_test_imp = pd.DataFrame(imputer.transform(X_test_c), columns=X_test_c.columns)
else:
    X_test_imp = pd.DataFrame(columns=X_test_c.columns)

rank_idx = stratified_cap_rows(y_train, FEATURE_SELECTION_MAX_ROWS)
X_rank = X_train_imp.iloc[rank_idx].reset_index(drop=True)
y_rank = y_train[rank_idx]

print("Features after variance filter:", len(base_keep))
print("Features after correlation prune:", len(corr_keep))
print("Rows used for ranking:", f"{len(X_rank):,}")

mi = mutual_info_classif(X_rank, y_rank, random_state=RANDOM_STATE)
mi_rank = pd.Series(mi, index=X_rank.columns).sort_values(ascending=False)

et_ranker = ExtraTreesClassifier(
    n_estimators=400,
    criterion="entropy",
    max_features="sqrt",
    min_samples_leaf=2,
    min_samples_split=4,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
et_ranker.fit(X_rank, y_rank)
et_rank = pd.Series(et_ranker.feature_importances_, index=X_rank.columns).sort_values(ascending=False)

xgb_rank = pd.Series(dtype=float)
if HAVE_XGBOOST:
    xgb_ranker = XGBClassifier(
        n_estimators=350,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        min_child_weight=2,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=0,
    )
    xgb_ranker.fit(X_rank, y_rank)
    xgb_rank = pd.Series(xgb_ranker.feature_importances_, index=X_rank.columns).sort_values(ascending=False)

perm_rank = pd.Series(dtype=float)
if RUN_PERMUTATION_IMPORTANCE:
    perm_idx = stratified_cap_rows(y_rank, PERMUTATION_MAX_ROWS)
    perm_candidates = list(dict.fromkeys(
        mi_rank.index.tolist()[:64] +
        et_rank.index.tolist()[:64] +
        xgb_rank.index.tolist()[:64]
    )) or list(X_rank.columns[:64])

    perm_model = ExtraTreesClassifier(
        n_estimators=300,
        criterion="entropy",
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    perm_model.fit(X_rank.iloc[perm_idx][perm_candidates], y_rank[perm_idx])
    perm = permutation_importance(
        perm_model,
        X_rank.iloc[perm_idx][perm_candidates],
        y_rank[perm_idx],
        n_repeats=PERMUTATION_REPEATS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    perm_rank = pd.Series(perm.importances_mean, index=perm_candidates).sort_values(ascending=False)

score = (
    rank_series_to_score(X_train_imp.columns, mi_rank.index.tolist(), TOP_MI, 1.0)
    + rank_series_to_score(X_train_imp.columns, et_rank.index.tolist(), TOP_ET, 1.15)
    + rank_series_to_score(X_train_imp.columns, xgb_rank.index.tolist(), TOP_XGB, 1.15)
    + rank_series_to_score(X_train_imp.columns, perm_rank.index.tolist(), TOP_PERM, 1.25)
).sort_values(ascending=False)

mandatory_feature_hints = [
    "x",
    "y",
    "z",
    "hag_terrain_xy_q10_k32",
    "hag_terrain_xy_q10_k64",
    "terrain_xy_relief_k32",
    "geom_planarity_k16",
    "geom_planarity_k32",
    "geom_sphericity_k16",
    "geom_surface_variation_k16",
    "geom_surface_variation_k32",
    "geom_verticality_k16",
    "geom_verticality_k32",
    "geom_linearity_k16",
    "geom_linearity_k32",
    "geom_anisotropy_k16",
    "geom_anisotropy_k32",
    "geom_eigenentropy_k16",
    "geom_eigenentropy_k32",
    "geom_normal_z_abs_k16",
    "geom_normal_z_abs_k32",
    "is_first_return",
    "is_last_return",
    "return_fraction",
    "return_number",
    "rgb_r_norm",
    "rgb_g_norm",
    "rgb_b_norm",
    "geom_adapt_best_k",
    "geom_adapt_planarity",
    "geom_adapt_sphericity",
    "geom_adapt_anisotropy",
    "geom_adapt_omnivariance",
    "geom_adapt_eigenentropy",
    "geom_adapt_surface_variation",
    "geom_adapt_normal_z_abs",
    "geom_adapt_verticality",
    "geom_adapt_z_std",
    "geom_adapt_z_range",
]
mandatory_features = [f for f in mandatory_feature_hints if f in X_train_imp.columns]

selected_features = []
for f in mandatory_features + score.index.tolist():
    if f not in selected_features:
        selected_features.append(f)
    if len(selected_features) >= min(TOP_CONSENSUS, len(score)):
        break

feature_ranking_df = pd.DataFrame(
    {
        "feature": score.index,
        "consensus_score": score.values,
        "mi_score": [float(mi_rank.get(f, 0.0)) for f in score.index],
        "et_score": [float(et_rank.get(f, 0.0)) for f in score.index],
        "xgb_score": [float(xgb_rank.get(f, 0.0)) for f in score.index],
        "perm_score": [float(perm_rank.get(f, 0.0)) for f in score.index],
        "is_mandatory": [f in mandatory_features for f in score.index],
    }
)
feature_ranking_df.to_csv(REPORT_DIR / "feature_ranking.csv", index=False)

print("Selected feature count:", len(selected_features))
print("Mandatory features retained:", mandatory_features)

family_summary = pd.Series(
    [
        "terrain" if "terrain_xy" in f else
        "geometry" if f.startswith("geom_") else
        "coordinates" if f in {"x", "y", "z"} else
        "grid" if "_g" in f or f.startswith("cell_") or f.startswith("nbr_") or f.startswith("hag_") else
        "intrinsic"
        for f in selected_features
    ]
).value_counts()

display(feature_ranking_df.head(40))
print("Selected feature families:")
display(family_summary.rename_axis("family").to_frame("count"))

X_train_sel = X_train_imp[selected_features].copy()
X_test_sel = X_test_imp[selected_features].copy()

extra_mandatory_hints = [
    "ground_proxy_z",
    "hag_ground_proxy",
    "ground_proxy_residual",
    "ground_proxy_slope_mag",
    "ground_proxy_curvature",
    "hag_ground_proxy_minus_terrain_q10_k32",
    "hag_ground_proxy_minus_cell_min_g1.0",
    "ground_proxy_slope_planarity_ratio",
]
for feature_name in extra_mandatory_hints:
    if feature_name in X_train_imp.columns and feature_name not in mandatory_features:
        mandatory_features.append(feature_name)

for coord_name in ("x", "y", "z"):
    if coord_name in X_train_imp.columns and coord_name not in mandatory_features:
        mandatory_features.append(coord_name)

selected_features = []
for f in mandatory_features + score.index.tolist():
    if f not in selected_features:
        selected_features.append(f)
    if len(selected_features) >= min(TOP_CONSENSUS + len(extra_mandatory_hints), len(X_train_imp.columns)):
        break

X_train_sel = X_train_imp[selected_features].copy()
X_test_sel = X_test_imp[selected_features].copy()

print("Selected feature count after ground-proxy keep-list:", len(selected_features))


Features after variance filter: 173
Features after correlation prune: 138
Rows used for ranking: 148,092
Selected feature count: 56
Mandatory features retained: ['x', 'y', 'z', 'hag_terrain_xy_q10_k32', 'terrain_xy_relief_k32', 'geom_planarity_k16', 'geom_planarity_k32', 'geom_sphericity_k16', 'geom_linearity_k16', 'geom_linearity_k32', 'geom_eigenentropy_k16', 'geom_eigenentropy_k32', 'geom_normal_z_abs_k16', 'geom_normal_z_abs_k32', 'is_first_return', 'return_fraction', 'return_number', 'geom_adapt_planarity', 'geom_adapt_sphericity', 'geom_adapt_omnivariance', 'geom_adapt_eigenentropy', 'geom_adapt_normal_z_abs', 'geom_adapt_z_std', 'geom_adapt_z_range']


,feature,consensus_score,mi_score,et_score,xgb_score,perm_score,is_mandatory
0,hag_terrain_xy_q10_k32,3.208480,0.745081,0.027089,0.226039,0.0,True
1,hag_terrain_xy_min_k16,3.204668,0.747553,0.028238,0.089869,0.0,False
2,hag_min_g16.0,3.171872,0.737297,0.036270,0.028119,0.0,False
3,hag_min_g8.0,3.161566,0.664870,0.028637,0.156970,0.0,False
4,hag_terrain_xy_q10_k16,3.102090,0.721907,0.020131,0.047636,0.0,False
5,nbr_ground_relief_g8.0,3.048307,0.571395,0.033240,0.022773,0.0,False
6,nbr_zrange_mean_g8.0,2.990761,0.749429,0.014892,0.035297,0.0,False
7,z,2.974338,0.605348,0.028962,0.007417,0.0,True
8,ground_slope_mag_g8.0,2.859197,0.676046,0.019103,0.004248,0.0,False
9,terrain_xy_relief_k16,2.838963,0.538676,0.015426,0.045373,0.0,False


Selected feature families:


,count
family,
grid,20
geometry,19
terrain,8
intrinsic,6
coordinates,3


Selected feature count after ground-proxy keep-list: 64


## Model families, spatial validation, neighborhood smoothing, and map cleanup

This section compares several non-neural classifiers, evaluates them under both ordinary and spatially separated validation, and then tests whether local post-processing improves the final class map. The main summary score is **macro F1**, which is the class-wise average of F1 and therefore gives each class equal weight rather than letting the largest class dominate the result ([scikit-learn F1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)). That is useful in airborne LiDAR classification because classes such as buildings, trees, ground, and cars are rarely present in equal numbers, yet performance on the rarer classes still matters for map quality.

### 1. Core model families

The first group of models uses **Extra Trees**, **Random Forest**, and **XGBoost**. These are all tree-based ensemble methods, but they behave differently.

**Random Forest** combines many decision trees and averages their predictions, which reduces variance and usually produces a stable classifier on heterogeneous feature tables ([Breiman, 2001](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf)). **Extra Trees** follows the same general ensemble idea, but injects more randomness into how splits are chosen, which can make it fast, robust, and competitive on nonlinear feature sets ([Geurts, Ernst, and Wehenkel, 2006](https://link.springer.com/content/pdf/10.1007/s10994-006-6226-1.pdf)). In airborne LiDAR and urban scene classification, tree ensembles have been widely used because they work well with mixed geometric, height, intensity, and neighborhood-derived predictors, and because they handle large tabular feature sets without requiring careful feature scaling ([Guo et al., 2011](https://www.sciencedirect.com/science/article/abs/pii/S0924271610000705); [Yan, Shaker, and El-Ashmawy, 2015](https://www.sciencedirect.com/science/article/abs/pii/S0034425714004374)).

The three Extra Trees presets in this notebook serve different practical roles:

- **`et`** is the main high-capacity baseline. It uses many trees and balanced class weighting so rare classes are not ignored.
- **`et_reg`** is a more regularized version. It uses bootstrap sampling, smaller feature subsets, and slightly larger minimum node sizes so the model is less eager to memorize narrow or noisy patterns.
- **`et_cons`** is the most conservative version. It pushes that regularization further, which can reduce scattered false positives at the cost of being less sensitive to smaller or weaker objects.
- **`et_car`** is a binary Extra Trees model that intentionally gives more weight to the car class so the model is more willing to call a point “car” instead of defaulting to a more common class.
- **`et_binary`** is a simpler binary Extra Trees model reused inside the hierarchical classifier for one-vs-rest steps such as ground vs non-ground.

The **XGBoost** model is a boosted-tree model rather than a bagged-tree ensemble. Instead of building many largely independent trees and averaging them, boosting fits trees sequentially so later trees concentrate on the remaining mistakes. In this code, `objective="multi:softprob"` returns per-class probabilities and `tree_method="hist"` uses the histogram-based tree builder, which is the standard efficient large-scale tree method in XGBoost ([XGBoost parameter documentation](https://xgboost.readthedocs.io/en/stable/parameter.html); [XGBoost tree method documentation](https://xgboost.readthedocs.io/en/stable/treemethod.html)). When GPU support is available, `device="cuda"` lets the same model family run on the GPU.

The optional **cuML random forest** is included as a GPU-oriented baseline. Its role here is mostly practical: it provides a fast tree-ensemble option when RAPIDS is already installed and the environment supports it.

### 2. Why multiscale and tuned variants are reasonable in LiDAR work

Airborne LiDAR classification usually depends heavily on how local neighborhoods are defined and how geometric summaries are extracted from them. A single neighborhood size is rarely ideal for every feature or every class. Reviews and comparative studies in point-cloud interpretation have shown that neighborhood choice strongly affects the distinctiveness of geometric features, and that multiscale or adaptively chosen neighborhoods are often more effective than a one-size-fits-all local support ([Weinmann et al., 2015](https://www.sciencedirect.com/science/article/pii/S0924271615000349); [Yan, Shaker, and El-Ashmawy, 2015](https://www.sciencedirect.com/science/article/abs/pii/S0034425714004374)).

That is the broader motivation for the tuned Extra Trees variants in this notebook. The goal is not to invent entirely different model families, but to give the same general tree-based framework slightly different operating styles so it can better handle the main confusion patterns in the dataset.

### 3. Two tuned Extra Trees variants

Two custom Extra Trees variants are defined because the four-class problem is not perfectly symmetric.

**`ExtraTreesSoftVoteTuned`** fits several Extra Trees models with slightly different regularization settings, averages their class probabilities, and then tunes a small set of class-specific scale factors on a held-out calibration split. In plain language, it asks: after averaging several reasonable models, should any class be nudged slightly upward or downward before taking the final winner? This is a lightweight calibration step intended to correct systematic overcalling or undercalling of certain classes while keeping the underlying models unchanged. Macro F1 is used for this tuning because the calibration should improve balance across classes rather than just the dominant class ([scikit-learn F1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)).

**`HierarchicalExtraTreesTuned`** breaks the problem into a short sequence of simpler binary decisions:

1. ground vs non-ground  
2. within non-ground, car vs not-car  
3. within the remaining non-car points, building vs tree  

This is motivated by the fact that airborne LiDAR classification is often not a flat, equally difficult separation problem. Ground filtering alone has a long literature because separating terrain from non-terrain has special geometric structure and downstream importance for normalized height modeling ([Yan, Shaker, and El-Ashmawy, 2015](https://www.sciencedirect.com/science/article/abs/pii/S0034425714004374); [Cao et al., 2024](https://www.mdpi.com/2072-4292/16/8/1443)). A staged classifier reflects that asymmetry: first solve the large terrain split, then treat the harder residual distinctions separately. The decision thresholds for ground, car, and building are tuned on held-out data to maximize macro F1.

### 4. Neighborhood smoothing of class probabilities

The function `tune_context_postprocess` adds a second stage after the base classifier. It only runs for models that can produce class probabilities. First, the model is fit on a training split. Then its probabilities on a held-out calibration split are smoothed using nearby points in space.

The smoothing step uses `k` nearest neighbors in a scaled coordinate system. Neighbor search is done either with RAPIDS cuML on the GPU or with SciPy’s `cKDTree` on the CPU. SciPy documents `cKDTree` as a structure for fast nearest-neighbor lookup, and `query` returns the nearest neighbors and their distances ([SciPy `cKDTree`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.cKDTree.html); [SciPy `cKDTree.query`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.cKDTree.query.html)).

The smoothing itself is simple:

- start with the model’s original per-class probabilities
- find the `k` nearest neighbors of each point
- average the neighbors’ probabilities, optionally giving closer points more weight
- blend that neighborhood average with the original probability using `alpha`

So `alpha` controls how much the model trusts local agreement. Low `alpha` means “mostly trust the original point-wise prediction.” Higher `alpha` means “lean more on what nearby points are doing.”

This is not an arbitrary add-on. Contextual classification has been a recurring theme in airborne LiDAR literature because purely point-independent predictions often produce locally inconsistent or speckled maps. Niemeyer et al. used a Random Forest plus conditional random field framework specifically to bring contextual information into urban airborne LiDAR classification, and Li et al. later proposed contextual label smoothing as a post-processing step to improve LiDAR classification accuracy by using neighborhood information more explicitly ([Niemeyer et al., 2014](https://www.sciencedirect.com/science/article/abs/pii/S0924271613002359); [Li, Liu, and Pfeifer, 2019](https://www.sciencedirect.com/science/article/abs/pii/S0924271618303241)). The smoothing used here is a simpler and lighter-weight version of the same general idea: encourage local consistency without replacing the original classifier.

### 5. Why there are two kinds of cross-validation

This block reports both **point-wise cross-validation** and **spatial cross-validation** because those answer different questions.

Point-wise cross-validation uses `StratifiedKFold`, which preserves class proportions across folds ([scikit-learn `StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)). This is useful for asking whether the model learns the class boundaries well under ordinary resampling.

But airborne LiDAR data are spatially autocorrelated. Nearby points tend to share structure, land cover, and acquisition conditions, so random folds can overestimate performance if very similar local patterns appear in both training and validation. Spatial cross-validation addresses that by grouping points into coarse `x`/`y` blocks and then splitting by those groups with either `GroupKFold` or `StratifiedGroupKFold` ([scikit-learn `GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html); [scikit-learn `StratifiedGroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)). In geospatial machine learning more broadly, spatial or blocked cross-validation is recommended when spatial dependence is present because random cross-validation can be optimistic about real transfer performance ([Sun and Huang, 2023](https://www.acsu.buffalo.edu/~yhu42/papers/2023_GeoAIHandbook_SpatialCV.pdf); [Wang et al., 2023](https://www.sciencedirect.com/science/article/pii/S1569843223001887)).

That is why the notebook reports both:
- **`cv_macro_f1_mean`**: random, class-balanced folds
- **`spatial_cv_macro_f1_mean`**: spatially separated folds

For map production, the second number is usually the more realistic one.

### 6. Small-island cleanup after prediction

The function `cleanup_point_predictions_by_components` is a light map-regularization step applied after prediction. It converts point labels into a coarse 2D grid in `x` and `y`, assigns each occupied grid cell its dominant class, identifies connected patches of the same class, and removes very small patches for selected classes by relabeling them according to the surrounding majority.

SciPy’s `ndimage.label` is used to identify connected components, and `generate_binary_structure` controls whether connectivity is 4-neighbor-like or 8-neighbor-like in two dimensions ([SciPy `label`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.label.html); [SciPy `generate_binary_structure`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.generate_binary_structure.html)).

In plain language, this says: if a tiny isolated building or car patch appears inside a much larger surrounding region, it is more likely to be point-level noise than a real object, so relabel it to match its surroundings.

This step is also consistent with the broader object-based tradition in LiDAR mapping. Object-based airborne LiDAR studies have shown that grouping or regularizing local predictions into more coherent spatial units can improve map usefulness and reduce salt-and-pepper artifacts, especially for urban scenes and land-cover mapping ([Antonarakis, Richards, and Brasington, 2008](https://www.sciencedirect.com/science/article/abs/pii/S0034425708000667); [Matikainen et al., 2017](https://www.sciencedirect.com/science/article/pii/S0924271616305305)). The cleanup used here is deliberately modest: it is not a full segmentation pipeline, just a practical way to suppress tiny isolated class islands.

### 7. Why some models are capped in training size

`model_fit_subset` allows selected models to be trained on a class-balanced subset rather than the entire training table. This is mainly a speed and memory control. Some models, and especially some post-processing stages, become expensive when the number of points grows very large. The cap keeps the comparison practical without discarding class balance.

This is especially relevant here because the notebook is not only fitting base models. It is also repeating cross-validation, calibration, neighborhood-based smoothing, and optional cleanup. Limiting fit size where needed allows those comparisons to stay tractable.

### 8. What the training loop records

For each model, the loop does four things:

1. fit and score point-wise cross-validation  
2. fit and score spatial cross-validation  
3. fit the model on the chosen training subset and optionally score the held-out test set  
4. optionally create additional result rows for:
   - the raw model
   - the same model after component cleanup
   - the same model after contextual probability smoothing
   - the same model after both smoothing and cleanup

This is why one underlying classifier can appear as several rows in the results table. The comparison is not only between model families, but also between different post-processing choices layered on top of those models.

### 9. How to read the result table

A good way to read the final table is:

- start with **spatial macro F1**, because it is the most realistic validation score here
- compare **test macro F1** and **weighted F1** to see whether a model is balanced across classes or mainly succeeding on the common classes
- check **context_enabled** and **component_cleanup_enabled** to see whether the gains come from the classifier itself or from the optional map-refinement steps
- inspect **calibration notes** for tuned soft-vote scales or hierarchical thresholds, because these show how the model needed to be adjusted to balance its class decisions

Overall, the design here is pragmatic and literature-aligned: use strong tree-based models on a multiscale LiDAR feature table, score them under both ordinary and spatial validation, then test whether simple local post-processing produces a cleaner and more stable class map without moving to a much heavier neural workflow.

### References

- [scikit-learn: `f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [Breiman, 2001, *Random Forests*](https://www.stat.berkeley.edu/~breiman/randomforest2001.pdf)
- [Geurts, Ernst, and Wehenkel, 2006, *Extremely randomized trees*](https://link.springer.com/content/pdf/10.1007/s10994-006-6226-1.pdf)
- [XGBoost parameter documentation](https://xgboost.readthedocs.io/en/stable/parameter.html)
- [XGBoost tree method documentation](https://xgboost.readthedocs.io/en/stable/treemethod.html)
- [Guo et al., 2011, *Relevance of airborne lidar and multispectral image data for urban scene classification using Random Forests*](https://www.sciencedirect.com/science/article/abs/pii/S0924271610000705)
- [Yan, Shaker, and El-Ashmawy, 2015, *Urban land cover classification using airborne LiDAR data: A review*](https://www.sciencedirect.com/science/article/abs/pii/S0034425714004374)
- [Weinmann et al., 2015, *Semantic point cloud interpretation based on optimal neighborhoods, relevant features and efficient classifiers*](https://www.sciencedirect.com/science/article/pii/S0924271615000349)
- [Niemeyer et al., 2014, *Contextual classification of lidar data and building object detection in urban areas*](https://www.sciencedirect.com/science/article/abs/pii/S0924271613002359)
- [Li, Liu, and Pfeifer, 2019, *Improving LiDAR classification accuracy by contextual label smoothing in post-processing*](https://www.sciencedirect.com/science/article/abs/pii/S0924271618303241)
- [scikit-learn: `StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [scikit-learn: `GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html)
- [scikit-learn: `StratifiedGroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html)
- [Sun and Huang, 2023, *Spatial cross-validation for GeoAI*](https://www.acsu.buffalo.edu/~yhu42/papers/2023_GeoAIHandbook_SpatialCV.pdf)
- [Wang et al., 2023, *A new cross-validation method to evaluate geospatial machine learning models*](https://www.sciencedirect.com/science/article/pii/S1569843223001887)
- [SciPy: `cKDTree`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.cKDTree.html)
- [SciPy: `cKDTree.query`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.cKDTree.query.html)
- [SciPy: `ndimage.label`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.label.html)
- [SciPy: `generate_binary_structure`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.generate_binary_structure.html)
- [Antonarakis, Richards, and Brasington, 2008, *Object-based land cover classification using airborne LiDAR*](https://www.sciencedirect.com/science/article/abs/pii/S0034425708000667)
- [Matikainen et al., 2017, *Object-based analysis of multispectral airborne laser scanner data for land cover classification and map updating*](https://www.sciencedirect.com/science/article/pii/S0924271616305305)
- [Cao et al., 2024, *A Multiscale Filtering Method for Airborne LiDAR Data*](https://www.mdpi.com/2072-4292/16/8/1443)

In [8]:
def _make_et_config(name: str):
    if name == "et":
        return ExtraTreesClassifier(
            n_estimators=800,
            criterion="entropy",
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if name == "et_reg":
        return ExtraTreesClassifier(
            n_estimators=1000,
            criterion="entropy",
            bootstrap=True,
            max_samples=0.85,
            max_features="sqrt",
            min_samples_leaf=2,
            min_samples_split=6,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if name == "et_cons":
        return ExtraTreesClassifier(
            n_estimators=1200,
            criterion="entropy",
            bootstrap=True,
            max_samples=0.75,
            max_features=0.45,
            min_samples_leaf=4,
            min_samples_split=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if name == "et_car":
        return ExtraTreesClassifier(
            n_estimators=900,
            criterion="entropy",
            bootstrap=True,
            max_samples=0.9,
            max_features="sqrt",
            min_samples_leaf=2,
            min_samples_split=4,
            class_weight={0: 1.0, 1: 3.0},
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if name == "et_binary":
        return ExtraTreesClassifier(
            n_estimators=900,
            criterion="entropy",
            bootstrap=True,
            max_samples=0.9,
            max_features="sqrt",
            min_samples_leaf=2,
            min_samples_split=4,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise ValueError(f"Unknown ET config: {name}")


def _make_xgb_classifier():
    params = dict(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.8,
        min_child_weight=2,
        reg_lambda=2.0,
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=0,
    )
    if USE_GPU_XGBOOST_IF_AVAILABLE:
        params["device"] = "cuda"
    return XGBClassifier(**params)


def _make_cuml_rf_classifier():
    return cuRFClassifier(
        n_estimators=600,
        max_features=1.0,
        max_depth=20,
        n_bins=128,
        random_state=RANDOM_STATE,
    )


def _scaled_argmax_predict(proba: np.ndarray, scales: Sequence[float]) -> np.ndarray:
    scaled = np.asarray(proba, dtype=np.float64) * np.asarray(scales, dtype=np.float64)[None, :]
    return np.argmax(scaled, axis=1).astype(np.int32)


def _to_scaled_spatial_coords(
    X_df: pd.DataFrame,
    spatial_columns: Sequence[str],
    spatial_scales: Sequence[float],
) -> np.ndarray:
    coords = X_df.loc[:, list(spatial_columns)].to_numpy(dtype=np.float32, copy=True)
    scales = np.asarray(spatial_scales, dtype=np.float32)
    return coords * scales[None, :]


def _query_neighbors_scaled(coords_scaled: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    n = int(len(coords_scaled))
    if n == 0:
        return np.zeros((0, 0), dtype=np.float32), np.zeros((0, 0), dtype=np.int64)

    k = int(max(1, min(int(k), max(1, n - 1))))
    query_k = min(n, k + 1)

    if USE_GPU_NEIGHBOR_SEARCH_IF_AVAILABLE and HAVE_CUML:
        nn = cuNearestNeighbors(n_neighbors=query_k)
        nn.fit(coords_scaled)
        dists, idx = nn.kneighbors(coords_scaled)
        dists = np.asarray(dists, dtype=np.float32)
        idx = np.asarray(idx, dtype=np.int64)
    else:
        tree = cKDTree(coords_scaled)
        dists, idx = tree.query(coords_scaled, k=query_k, workers=-1)
        if query_k == 1:
            dists = dists[:, None]
            idx = idx[:, None]
        dists = np.asarray(dists, dtype=np.float32)
        idx = np.asarray(idx, dtype=np.int64)

    if query_k > 1:
        return dists[:, 1:], idx[:, 1:]
    return dists[:, :0], idx[:, :0]


def smooth_probabilities_with_coords(
    proba: np.ndarray,
    coords_scaled: np.ndarray,
    k: int,
    alpha: float,
    use_distance_weights: bool = True,
    precomputed_neighbors: Optional[Tuple[np.ndarray, np.ndarray]] = None,
) -> np.ndarray:
    proba = np.asarray(proba, dtype=np.float32)
    n = int(len(proba))
    if n == 0 or k <= 0 or alpha <= 0.0:
        return proba

    if precomputed_neighbors is None:
        dists, idx = _query_neighbors_scaled(coords_scaled, k)
    else:
        dists, idx = precomputed_neighbors

    if idx.size == 0:
        return proba

    neigh_proba = proba[idx]
    if use_distance_weights:
        weights = 1.0 / np.maximum(dists, 1e-3)
        weights = weights.astype(np.float32, copy=False)
        weights = weights / np.maximum(weights.sum(axis=1, keepdims=True), 1e-6)
        neigh_mean = (neigh_proba * weights[..., None]).sum(axis=1).astype(np.float32)
    else:
        neigh_mean = neigh_proba.mean(axis=1).astype(np.float32)

    alpha = float(alpha)
    smoothed = ((1.0 - alpha) * proba + alpha * neigh_mean).astype(np.float32)
    denom = np.clip(smoothed.sum(axis=1, keepdims=True), 1e-12, None)
    return smoothed / denom




def fresh_model_instance(model):
    try:
        return clone(model)
    except Exception:
        return copy.deepcopy(model)


def spatial_block_groups_from_xy(
    X_df: pd.DataFrame,
    block_size: float = SPATIAL_CV_BLOCK_SIZE,
) -> np.ndarray:
    if "x" not in X_df.columns or "y" not in X_df.columns:
        raise ValueError("Spatial block grouping requires x and y columns.")
    ix = np.floor(X_df["x"].to_numpy(dtype=np.float32) / float(block_size)).astype(np.int64)
    iy = np.floor(X_df["y"].to_numpy(dtype=np.float32) / float(block_size)).astype(np.int64)
    return pack_xy(ix, iy)


def iter_spatial_splits(
    X_df: pd.DataFrame,
    y: np.ndarray,
    n_splits: int = SPATIAL_CV_FOLDS,
    block_size: float = SPATIAL_CV_BLOCK_SIZE,
):
    groups = spatial_block_groups_from_xy(X_df, block_size=block_size)
    if HAVE_STRATIFIED_GROUP_KFOLD:
        splitter = StratifiedGroupKFold(
            n_splits=int(n_splits),
            shuffle=True,
            random_state=RANDOM_STATE,
        )
        return splitter.split(X_df, y, groups=groups)

    splitter = GroupKFold(n_splits=int(n_splits))
    return splitter.split(X_df, y, groups=groups)


def cleanup_point_predictions_by_components(
    X_df: pd.DataFrame,
    pred: np.ndarray,
    grid_size: float = COMPONENT_GRID_SIZE,
    min_cells_by_class: Optional[Dict[int, int]] = None,
    connectivity: int = COMPONENT_CONNECTIVITY,
) -> np.ndarray:
    if min_cells_by_class is None:
        min_cells_by_class = COMPONENT_MIN_CELLS_BY_CLASS

    if len(pred) == 0 or "x" not in X_df.columns or "y" not in X_df.columns:
        return np.asarray(pred, dtype=np.int32)

    xy = X_df.loc[:, ["x", "y"]].to_numpy(dtype=np.float32, copy=False)
    pred = np.asarray(pred, dtype=np.int32, copy=False)

    ix = np.floor(xy[:, 0] / float(grid_size)).astype(np.int64)
    iy = np.floor(xy[:, 1] / float(grid_size)).astype(np.int64)
    keys = pack_xy(ix, iy)

    uniq_keys, inv = np.unique(keys, return_inverse=True)
    n_classes = max(len(CLASS_NAMES), int(pred.max()) + 1)

    cell_class_counts = np.zeros((len(uniq_keys), n_classes), dtype=np.int32)
    np.add.at(cell_class_counts, (inv, pred), 1)
    cell_label = np.argmax(cell_class_counts, axis=1).astype(np.int32)

    cell_ix, cell_iy = unpack_xy(uniq_keys)
    min_ix = int(cell_ix.min())
    min_iy = int(cell_iy.min())
    rows = (cell_iy - min_iy).astype(np.int64, copy=False)
    cols = (cell_ix - min_ix).astype(np.int64, copy=False)

    grid = np.full((int(rows.max()) + 1, int(cols.max()) + 1), -1, dtype=np.int32)
    grid[rows, cols] = cell_label

    structure = ndimage.generate_binary_structure(2, 2 if int(connectivity) >= 8 else 1)

    for class_id, min_cells in sorted(min_cells_by_class.items()):
        min_cells = int(min_cells)
        if min_cells <= 1:
            continue

        mask = grid == int(class_id)
        if not np.any(mask):
            continue

        labels, n_labels = ndimage.label(mask, structure=structure)
        if n_labels == 0:
            continue

        sizes = np.bincount(labels.ravel(), minlength=n_labels + 1)
        for label_id in range(1, n_labels + 1):
            if int(sizes[label_id]) >= min_cells:
                continue

            comp_mask = labels == label_id
            border = ndimage.binary_dilation(comp_mask, structure=structure) & ~comp_mask
            neighbor_classes = grid[border]
            neighbor_classes = neighbor_classes[neighbor_classes >= 0]
            if len(neighbor_classes) == 0:
                continue

            replacement = int(np.bincount(neighbor_classes, minlength=n_classes).argmax())
            grid[comp_mask] = replacement

    cleaned_cell_labels = grid[rows, cols].astype(np.int32, copy=False)
    return cleaned_cell_labels[inv].astype(np.int32, copy=False)


def tune_context_postprocess(
    model,
    X_df: pd.DataFrame,
    y: np.ndarray,
    k_grid: Sequence[int] = CONTEXT_K_GRID,
    alpha_grid: Sequence[float] = CONTEXT_ALPHA_GRID,
    spatial_columns: Sequence[str] = CONTEXT_SPATIAL_COLUMNS,
    spatial_scales: Sequence[float] = CONTEXT_SPATIAL_SCALES,
    use_distance_weights: bool = CONTEXT_USE_DISTANCE_WEIGHTS,
    calibration_fraction: float = CALIBRATION_FRACTION,
):
    if not hasattr(model, "predict_proba"):
        return None

    if not set(spatial_columns).issubset(X_df.columns):
        return None

    X_fit = X_df.reset_index(drop=True)
    y_fit = np.asarray(y, dtype=np.int32)

    X_tr, X_cal, y_tr, y_cal = train_test_split(
        X_fit,
        y_fit,
        test_size=float(calibration_fraction),
        stratify=y_fit,
        random_state=RANDOM_STATE,
    )

    m = fresh_model_instance(model)
    m.fit(X_tr, y_tr)
    cal_proba = np.asarray(m.predict_proba(X_cal), dtype=np.float32)
    coords_scaled = _to_scaled_spatial_coords(X_cal, spatial_columns, spatial_scales)

    best = None
    best_score = -np.inf
    for k in k_grid:
        neighbor_cache = _query_neighbors_scaled(coords_scaled, int(k))
        for alpha in alpha_grid:
            smoothed = smooth_probabilities_with_coords(
                cal_proba,
                coords_scaled,
                k=int(k),
                alpha=float(alpha),
                use_distance_weights=use_distance_weights,
                precomputed_neighbors=neighbor_cache,
            )
            pred = np.argmax(smoothed, axis=1).astype(np.int32)
            score = f1_score(y_cal, pred, average="macro")
            if score > best_score:
                best_score = float(score)
                best = {
                    "k": int(k),
                    "alpha": float(alpha),
                    "spatial_columns": tuple(spatial_columns),
                    "spatial_scales": tuple(float(v) for v in spatial_scales),
                    "use_distance_weights": bool(use_distance_weights),
                    "calibration_macro_f1": float(score),
                }

    return best


def predict_with_context_postprocess(
    model,
    X_df: pd.DataFrame,
    context_params: Optional[Dict[str, object]] = None,
    return_proba: bool = False,
):
    if context_params is None:
        if return_proba and hasattr(model, "predict_proba"):
            return np.asarray(model.predict_proba(X_df), dtype=np.float32)
        pred = np.asarray(model.predict(X_df), dtype=np.int32)
        return pred

    proba = np.asarray(model.predict_proba(X_df), dtype=np.float32)
    coords_scaled = _to_scaled_spatial_coords(
        X_df,
        context_params["spatial_columns"],
        context_params["spatial_scales"],
    )
    smoothed = smooth_probabilities_with_coords(
        proba,
        coords_scaled,
        k=int(context_params["k"]),
        alpha=float(context_params["alpha"]),
        use_distance_weights=bool(context_params["use_distance_weights"]),
    )
    if return_proba:
        return smoothed
    return np.argmax(smoothed, axis=1).astype(np.int32)


class ExtraTreesSoftVoteTuned(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        base_model_keys=("et_reg", "et", "et_cons"),
        calibration_fraction=CALIBRATION_FRACTION,
        random_state=RANDOM_STATE,
        scale_grid=None,
    ):
        self.base_model_keys = base_model_keys
        self.calibration_fraction = calibration_fraction
        self.random_state = random_state
        self.scale_grid = scale_grid

    def _default_scale_grid(self):
        return list(product(
            [1.00, 1.05, 1.10, 1.15, 1.20],
            [0.95, 1.00, 1.05],
            [0.92, 0.97, 1.00],
            [1.20, 1.45, 1.75, 2.10, 2.50],
        ))

    def _fit_models(self, X, y):
        models = []
        for key in tuple(self.base_model_keys):
            m = _make_et_config(key)
            m.fit(X, y)
            models.append(m)
        return models

    def _avg_proba(self, models, X):
        probs = [m.predict_proba(X) for m in models]
        return np.mean(np.stack(probs, axis=0), axis=0)

    def fit(self, X, y):
        X_fit = X.reset_index(drop=True) if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        y_fit = np.asarray(y, dtype=np.int32)

        cal_frac = float(self.calibration_fraction)
        rs = int(self.random_state)

        X_tr, X_cal, y_tr, y_cal = train_test_split(
            X_fit,
            y_fit,
            test_size=cal_frac,
            stratify=y_fit,
            random_state=rs,
        )

        cal_models = self._fit_models(X_tr, y_tr)
        cal_proba = self._avg_proba(cal_models, X_cal)

        grid = self.scale_grid
        if grid is None:
            grid = self._default_scale_grid()

        best = None
        best_score = -np.inf
        for scales in grid:
            pred = _scaled_argmax_predict(cal_proba, scales)
            score = f1_score(y_cal, pred, average="macro")
            if score > best_score:
                best_score = float(score)
                best = tuple(float(v) for v in scales)

        self.best_scales_ = best
        self.calibration_macro_f1_ = float(best_score)
        self.models_ = self._fit_models(X_fit, y_fit)
        self.classes_ = np.array(sorted(np.unique(y_fit)), dtype=np.int32)
        self.n_features_in_ = X_fit.shape[1]
        self.feature_names_in_ = np.asarray(X_fit.columns, dtype=object)
        return self

    def predict_proba(self, X):
        X_use = X.reset_index(drop=True) if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        proba = self._avg_proba(self.models_, X_use)
        scaled = proba * np.asarray(self.best_scales_, dtype=np.float64)[None, :]
        denom = np.clip(scaled.sum(axis=1, keepdims=True), 1e-12, None)
        return scaled / denom

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1).astype(np.int32)


class HierarchicalExtraTreesTuned(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        calibration_fraction=CALIBRATION_FRACTION,
        random_state=RANDOM_STATE,
        ground_thresholds=(0.45, 0.50, 0.55, 0.60),
        car_thresholds=(0.20, 0.25, 0.30, 0.35, 0.40),
        building_thresholds=(0.45, 0.50, 0.55),
    ):
        self.calibration_fraction = calibration_fraction
        self.random_state = random_state
        self.ground_thresholds = ground_thresholds
        self.car_thresholds = car_thresholds
        self.building_thresholds = building_thresholds

    def _fit_components(self, X, y):
        ground_model = _make_et_config("et_binary")
        ground_model.fit(X, (y == 2).astype(np.int32))

        non_ground_mask = y != 2
        car_model = _make_et_config("et_car")
        car_model.fit(X.loc[non_ground_mask], (y[non_ground_mask] == 3).astype(np.int32))

        bt_mask = np.isin(y, [0, 1])
        bt_model = _make_et_config("et_binary")
        bt_model.fit(X.loc[bt_mask], (y[bt_mask] == 0).astype(np.int32))

        return ground_model, car_model, bt_model

    def _compose_proba(self, X):
        p_ground = self.ground_model_.predict_proba(X)[:, 1]
        p_nonground = np.clip(1.0 - p_ground, 0.0, 1.0)

        p_car_given_ng = self.car_model_.predict_proba(X)[:, 1]
        p_notcar_given_ng = np.clip(1.0 - p_car_given_ng, 0.0, 1.0)

        p_building_given_bt = self.bt_model_.predict_proba(X)[:, 1]
        p_tree_given_bt = np.clip(1.0 - p_building_given_bt, 0.0, 1.0)

        p_car = p_nonground * p_car_given_ng
        p_building = p_nonground * p_notcar_given_ng * p_building_given_bt
        p_tree = p_nonground * p_notcar_given_ng * p_tree_given_bt

        proba = np.stack([p_building, p_tree, p_ground, p_car], axis=1)
        denom = np.clip(proba.sum(axis=1, keepdims=True), 1e-12, None)
        return proba / denom

    def _predict_with_thresholds(self, X, t_ground, t_car, t_building):
        p_ground = self.ground_model_.predict_proba(X)[:, 1]
        pred = np.full(len(X), 1, dtype=np.int32)

        ground_mask = p_ground >= t_ground
        pred[ground_mask] = 2

        remaining = ~ground_mask
        if np.any(remaining):
            X_rem = X.loc[remaining]
            p_car = self.car_model_.predict_proba(X_rem)[:, 1]
            car_mask = p_car >= t_car

            rem_idx = np.flatnonzero(remaining)
            pred[rem_idx[car_mask]] = 3

            bt_idx = rem_idx[~car_mask]
            if len(bt_idx) > 0:
                X_bt = X.iloc[bt_idx]
                p_building = self.bt_model_.predict_proba(X_bt)[:, 1]
                pred[bt_idx] = np.where(p_building >= t_building, 0, 1).astype(np.int32)

        return pred

    def fit(self, X, y):
        X_fit = X.reset_index(drop=True) if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        y_fit = np.asarray(y, dtype=np.int32)

        cal_frac = float(self.calibration_fraction)
        rs = int(self.random_state)
        ground_thresholds = tuple(float(v) for v in self.ground_thresholds)
        car_thresholds = tuple(float(v) for v in self.car_thresholds)
        building_thresholds = tuple(float(v) for v in self.building_thresholds)

        X_tr, X_cal, y_tr, y_cal = train_test_split(
            X_fit,
            y_fit,
            test_size=cal_frac,
            stratify=y_fit,
            random_state=rs,
        )

        self.ground_model_, self.car_model_, self.bt_model_ = self._fit_components(X_tr, y_tr)

        best = None
        best_score = -np.inf
        for tg, tc, tb in product(ground_thresholds, car_thresholds, building_thresholds):
            pred = self._predict_with_thresholds(X_cal, tg, tc, tb)
            score = f1_score(y_cal, pred, average="macro")
            if score > best_score:
                best_score = float(score)
                best = (float(tg), float(tc), float(tb))

        self.best_thresholds_ = best
        self.calibration_macro_f1_ = float(best_score)
        self.ground_model_, self.car_model_, self.bt_model_ = self._fit_components(X_fit, y_fit)
        self.classes_ = np.array([0, 1, 2, 3], dtype=np.int32)
        self.n_features_in_ = X_fit.shape[1]
        self.feature_names_in_ = np.asarray(X_fit.columns, dtype=object)
        return self

    def predict_proba(self, X):
        X_use = X.reset_index(drop=True) if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        return self._compose_proba(X_use)

    def predict(self, X):
        X_use = X.reset_index(drop=True) if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        tg, tc, tb = self.best_thresholds_
        return self._predict_with_thresholds(X_use, tg, tc, tb)


def model_fit_subset(name: str, X_df: pd.DataFrame, y: np.ndarray):
    max_rows = MODEL_FIT_SAMPLE_CAPS.get(name)
    if max_rows is None or len(X_df) <= max_rows:
        return X_df, y
    keep_idx = stratified_cap_rows(y, int(max_rows))
    return X_df.iloc[keep_idx].reset_index(drop=True), y[keep_idx]


all_models = {}

all_models["ExtraTrees"] = Pipeline([
    ("identity", "passthrough"),
    ("clf", _make_et_config("et")),
])

all_models["ExtraTreesRegularized"] = Pipeline([
    ("identity", "passthrough"),
    ("clf", _make_et_config("et_reg")),
])

all_models["ExtraTreesConservative"] = Pipeline([
    ("identity", "passthrough"),
    ("clf", _make_et_config("et_cons")),
])

all_models["RandomForest"] = Pipeline([
    ("identity", "passthrough"),
    ("clf", RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=4,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

all_models["ExtraTreesSoftVoteTuned"] = ExtraTreesSoftVoteTuned(
    calibration_fraction=CALIBRATION_FRACTION,
    random_state=RANDOM_STATE,
)

all_models["HierarchicalExtraTreesTuned"] = HierarchicalExtraTreesTuned(
    calibration_fraction=CALIBRATION_FRACTION,
    random_state=RANDOM_STATE,
)

if HAVE_XGBOOST:
    all_models["XGBoost"] = Pipeline([
        ("identity", "passthrough"),
        ("clf", _make_xgb_classifier()),
    ])

if HAVE_CUML and USE_CUML_RF_IF_AVAILABLE:
    all_models["cuMLRandomForest"] = _make_cuml_rf_classifier()

models = {name: model for name, model in all_models.items() if name in MODEL_NAMES_TO_RUN}
if not models:
    raise ValueError(f"No models selected. Available: {list(all_models)} | requested: {MODEL_NAMES_TO_RUN}")

have_test = len(X_test_sel) > 0 and len(y_test) > 0


pointwise_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
trained_models = {}

for name, model in models.items():
    print("=" * 80)
    print("Training:", name)
    fit_X_full, fit_y_full = model_fit_subset(name, X_train_sel, y_train)
    if len(fit_X_full) != len(X_train_sel):
        print(f"Using stratified fit cap for {name}: {len(fit_X_full):,} rows")

    pointwise_fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(pointwise_cv.split(fit_X_full, fit_y_full), start=1):
        m = fresh_model_instance(model)
        m.fit(fit_X_full.iloc[tr_idx], fit_y_full[tr_idx])
        pred_va = m.predict(fit_X_full.iloc[va_idx])
        f1m = f1_score(fit_y_full[va_idx], pred_va, average="macro")
        pointwise_fold_scores.append(float(f1m))
        print(f"Pointwise fold {fold}: macro-F1 = {f1m:.4f}")

    spatial_fold_scores = []
    if RUN_SPATIAL_CV and {"x", "y"}.issubset(fit_X_full.columns):
        try:
            for fold, (tr_idx, va_idx) in enumerate(iter_spatial_splits(fit_X_full, fit_y_full), start=1):
                m = fresh_model_instance(model)
                m.fit(fit_X_full.iloc[tr_idx], fit_y_full[tr_idx])
                pred_va = m.predict(fit_X_full.iloc[va_idx])
                f1m = f1_score(fit_y_full[va_idx], pred_va, average="macro")
                spatial_fold_scores.append(float(f1m))
                print(f"Spatial fold {fold}: macro-F1 = {f1m:.4f}")
        except Exception as exc:
            print(f"Spatial CV skipped for {name}: {exc}")

    t0 = time.perf_counter()
    model.fit(fit_X_full, fit_y_full)
    train_time = time.perf_counter() - t0

    row = {
        "strategy": "KitchenSink",
        "model": name,
        "selected_feature_count": int(len(selected_features)),
        "selected_features": list(selected_features),
        "fit_rows": int(len(fit_X_full)),
        "cv_macro_f1_mean": float(np.mean(pointwise_fold_scores)),
        "cv_macro_f1_std": float(np.std(pointwise_fold_scores)),
        "spatial_cv_macro_f1_mean": float(np.mean(spatial_fold_scores)) if spatial_fold_scores else np.nan,
        "spatial_cv_macro_f1_std": float(np.std(spatial_fold_scores)) if spatial_fold_scores else np.nan,
        "train_time_sec": float(train_time),
        "test_time_sec": np.nan,
        "accuracy": np.nan,
        "kappa": np.nan,
        "f1_macro": np.nan,
        "weighted_f1": np.nan,
        "calibration_macro_f1": getattr(model, "calibration_macro_f1_", np.nan),
        "calibration_notes": (
            str(getattr(model, "best_scales_", getattr(model, "best_thresholds_", "")))
            if hasattr(model, "best_scales_") or hasattr(model, "best_thresholds_")
            else ""
        ),
        "context_enabled": False,
        "component_cleanup_enabled": False,
    }

    pred_test = None
    if have_test:
        t0 = time.perf_counter()
        pred_test = np.asarray(model.predict(X_test_sel), dtype=np.int32)
        test_time = time.perf_counter() - t0
        row["test_time_sec"] = float(test_time)
        row["accuracy"] = float(accuracy_score(y_test, pred_test))
        row["f1_macro"] = float(f1_score(y_test, pred_test, average="macro"))
        row["weighted_f1"] = float(f1_score(y_test, pred_test, average="weighted"))
        row["kappa"] = float(cohen_kappa_score(y_test, pred_test))

    cv_rows.append(row)
    trained_models[name] = {
        "model": model,
        "context_params": None,
        "cleanup_params": None,
    }

    if RUN_COMPONENT_CLEANUP and have_test and pred_test is not None and {"x", "y"}.issubset(X_test_sel.columns):
        pred_test_clean = cleanup_point_predictions_by_components(X_test_sel, pred_test)
        clean_row = dict(row)
        clean_row["model"] = f"{name}{COMPONENT_SUFFIX}"
        clean_row["accuracy"] = float(accuracy_score(y_test, pred_test_clean))
        clean_row["f1_macro"] = float(f1_score(y_test, pred_test_clean, average="macro"))
        clean_row["weighted_f1"] = float(f1_score(y_test, pred_test_clean, average="weighted"))
        clean_row["kappa"] = float(cohen_kappa_score(y_test, pred_test_clean))
        clean_row["component_cleanup_enabled"] = True
        cv_rows.append(clean_row)
        trained_models[clean_row["model"]] = {
            "model": model,
            "context_params": None,
            "cleanup_params": {
                "grid_size": float(COMPONENT_GRID_SIZE),
                "connectivity": int(COMPONENT_CONNECTIVITY),
                "min_cells_by_class": dict(COMPONENT_MIN_CELLS_BY_CLASS),
            },
        }

    if RUN_CONTEXTUAL_POSTPROCESS and have_test and hasattr(model, "predict_proba") and set(CONTEXT_SPATIAL_COLUMNS).issubset(fit_X_full.columns):
        context_params = tune_context_postprocess(model, fit_X_full, fit_y_full)
        if context_params is not None:
            t0 = time.perf_counter()
            pred_test_ctx = predict_with_context_postprocess(model, X_test_sel, context_params=context_params)
            test_time_ctx = time.perf_counter() - t0

            ctx_row = dict(row)
            ctx_row["model"] = f"{name}{CONTEXT_SUFFIX}"
            ctx_row["test_time_sec"] = float(test_time_ctx)
            ctx_row["accuracy"] = float(accuracy_score(y_test, pred_test_ctx))
            ctx_row["f1_macro"] = float(f1_score(y_test, pred_test_ctx, average="macro"))
            ctx_row["weighted_f1"] = float(f1_score(y_test, pred_test_ctx, average="weighted"))
            ctx_row["kappa"] = float(cohen_kappa_score(y_test, pred_test_ctx))
            ctx_row["calibration_macro_f1"] = float(context_params["calibration_macro_f1"])
            ctx_row["calibration_notes"] = str({
                "k": context_params["k"],
                "alpha": context_params["alpha"],
                "spatial_columns": context_params["spatial_columns"],
                "spatial_scales": context_params["spatial_scales"],
                "distance_weights": context_params["use_distance_weights"],
            })
            ctx_row["context_enabled"] = True

            cv_rows.append(ctx_row)
            trained_models[ctx_row["model"]] = {
                "model": model,
                "context_params": context_params,
                "cleanup_params": None,
            }
            print("Context smoothing tuned:", ctx_row["model"], ctx_row["calibration_notes"])

            if RUN_COMPONENT_CLEANUP and {"x", "y"}.issubset(X_test_sel.columns):
                pred_test_ctx_clean = cleanup_point_predictions_by_components(X_test_sel, pred_test_ctx)
                ctx_clean_row = dict(ctx_row)
                ctx_clean_row["model"] = f"{ctx_row['model']}{COMPONENT_SUFFIX}"
                ctx_clean_row["accuracy"] = float(accuracy_score(y_test, pred_test_ctx_clean))
                ctx_clean_row["f1_macro"] = float(f1_score(y_test, pred_test_ctx_clean, average="macro"))
                ctx_clean_row["weighted_f1"] = float(f1_score(y_test, pred_test_ctx_clean, average="weighted"))
                ctx_clean_row["kappa"] = float(cohen_kappa_score(y_test, pred_test_ctx_clean))
                ctx_clean_row["component_cleanup_enabled"] = True
                cv_rows.append(ctx_clean_row)
                trained_models[ctx_clean_row["model"]] = {
                    "model": model,
                    "context_params": context_params,
                    "cleanup_params": {
                        "grid_size": float(COMPONENT_GRID_SIZE),
                        "connectivity": int(COMPONENT_CONNECTIVITY),
                        "min_cells_by_class": dict(COMPONENT_MIN_CELLS_BY_CLASS),
                    },
                }

cv_results_df = pd.DataFrame(cv_rows)
sort_cols = ["f1_macro", "spatial_cv_macro_f1_mean", "cv_macro_f1_mean"] if have_test else ["spatial_cv_macro_f1_mean", "cv_macro_f1_mean"]
cv_results_df = cv_results_df.sort_values(sort_cols, ascending=False, na_position="last").reset_index(drop=True)
cv_results_df.to_csv(REPORT_DIR / "model_results_cv_only.csv", index=False)
display(cv_results_df[[
    "strategy",
    "model",
    "fit_rows",
    "selected_feature_count",
    "cv_macro_f1_mean",
    "cv_macro_f1_std",
    "spatial_cv_macro_f1_mean",
    "spatial_cv_macro_f1_std",
    "train_time_sec",
    "test_time_sec",
    "accuracy",
    "f1_macro",
    "weighted_f1",
    "kappa",
    "context_enabled",
    "component_cleanup_enabled",
]])


Training: ExtraTrees
Pointwise fold 1: macro-F1 = 0.9586
Pointwise fold 2: macro-F1 = 0.9650
Pointwise fold 3: macro-F1 = 0.9650
Spatial fold 1: macro-F1 = 0.8839
Spatial fold 2: macro-F1 = 0.8993
Spatial fold 3: macro-F1 = 0.8567
Context smoothing tuned: ExtraTreesContextKNN {'k': 8, 'alpha': 0.2, 'spatial_columns': ('x', 'y', 'z'), 'spatial_scales': (1.0, 1.0, 2.0), 'distance_weights': True}
Training: ExtraTreesRegularized
Pointwise fold 1: macro-F1 = 0.9529
Pointwise fold 2: macro-F1 = 0.9590
Pointwise fold 3: macro-F1 = 0.9591
Spatial fold 1: macro-F1 = 0.8922
Spatial fold 2: macro-F1 = 0.9150
Spatial fold 3: macro-F1 = 0.8581
Context smoothing tuned: ExtraTreesRegularizedContextKNN {'k': 32, 'alpha': 0.2, 'spatial_columns': ('x', 'y', 'z'), 'spatial_scales': (1.0, 1.0, 2.0), 'distance_weights': True}
Training: ExtraTreesConservative
Pointwise fold 1: macro-F1 = 0.9509
Pointwise fold 2: macro-F1 = 0.9551
Pointwise fold 3: macro-F1 = 0.9553
Spatial fold 1: macro-F1 = 0.8974
Spatial 

KeyboardInterrupt: 

## Block summary: reporting, figures, export helpers, and full-cloud post-processing

Methods used in this block:
- save confusion matrices, classified LAS outputs, and 2D render previews
- export model bundles that preserve both context-smoothing and component-cleanup settings
- run full-cloud prediction in batches so the selected recipe can be replayed on the original LAS


In [9]:

from PIL import Image, ImageFilter

CLASS_PALETTE = {
    0: (230, 40, 70),
    1: (30, 170, 30),
    2: (80, 80, 80),
    3: (0, 170, 255),
}

VIEW_MAX_DIM = 2200
VIEW_MODE_FILTER_SIZE = 5
VIEW_MODE_FILTER_PASSES = 1
VIEW_BACKGROUND = (248, 248, 248)

def build_palette_lut_from_labels(palette, n_classes):
    lut = np.zeros((max(256, n_classes + 1), 3), dtype=np.uint8)
    for k, rgb in palette.items():
        lut[int(k)] = np.asarray(rgb, dtype=np.uint8)
    return lut


def _estimate_pixel_size(xa, ya, max_dim):
    min_x = float(np.min(xa))
    max_x = float(np.max(xa))
    min_y = float(np.min(ya))
    max_y = float(np.max(ya))
    dx = max(max_x - min_x, 1e-6)
    dy = max(max_y - min_y, 1e-6)
    pixel_size = max(dx / max_dim, dy / max_dim)
    pixel_size = max(pixel_size, 1e-6)
    width = int(np.ceil(dx / pixel_size)) + 1
    height = int(np.ceil(dy / pixel_size)) + 1
    return min_x, max_y, pixel_size, width, height


def save_view_raster_png(
    xyz,
    labels,
    out_path,
    view="top",
    max_dim=VIEW_MAX_DIM,
    mode_filter_size=VIEW_MODE_FILTER_SIZE,
    mode_filter_passes=VIEW_MODE_FILTER_PASSES,
    background_rgb=VIEW_BACKGROUND,
    palette=None,
):
    xyz = np.asarray(xyz, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)

    if xyz.shape[0] == 0:
        return None

    if view == "top":
        xa = xyz[:, 0]
        ya = xyz[:, 1]
    elif view == "side":
        xa = xyz[:, 0]
        ya = xyz[:, 2]
    else:
        raise ValueError(f"Unknown view: {view}")

    min_x, max_y, pixel_size, width, height = _estimate_pixel_size(xa, ya, max_dim=max_dim)

    col = np.floor((xa - min_x) / pixel_size).astype(np.int32)
    row = np.floor((max_y - ya) / pixel_size).astype(np.int32)

    mask = (row >= 0) & (row < height) & (col >= 0) & (col < width)
    row = row[mask]
    col = col[mask]
    labels = labels[mask]

    if row.size == 0:
        return None

    n_classes = len(CLASS_NAMES)
    flat = row.astype(np.int64) * width + col.astype(np.int64)

    counts = np.zeros((height * width, n_classes), dtype=np.uint32)
    np.add.at(counts, (flat, labels), 1)

    majority = counts.argmax(axis=1).astype(np.uint8).reshape(height, width)
    occupancy = counts.sum(axis=1).reshape(height, width) > 0

    class_img = Image.fromarray(majority, mode="L")
    for _ in range(int(mode_filter_passes)):
        class_img = class_img.filter(ImageFilter.ModeFilter(size=int(mode_filter_size)))
    majority = np.array(class_img, dtype=np.uint8, copy=True)
    majority[~occupancy] = 255

    lut = build_palette_lut_from_labels(palette or CLASS_PALETTE, n_classes)
    rgb = np.empty((height, width, 3), dtype=np.uint8)
    rgb[..., 0] = background_rgb[0]
    rgb[..., 1] = background_rgb[1]
    rgb[..., 2] = background_rgb[2]

    valid = majority != 255
    rgb[valid] = lut[majority[valid]]

    Image.fromarray(rgb, mode="RGB").save(out_path)
    return out_path


def save_confusion_matrix_png(cm, class_names, out_path, title):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    vmax = max(int(cm.max()), 1)
    thresh = vmax * 0.5
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                str(int(cm[i, j])),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=10,
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    return out_path


def build_subset_prediction_writer_header(input_header):
    header = copy.deepcopy(input_header)
    dim_names = [str(dim.name).lower() for dim in header.point_format.dimensions]
    extras = []
    if "id" not in dim_names:
        extras.append(laspy.ExtraBytesParams(name="Id", type=np.int64))
    if "true_label" not in dim_names:
        extras.append(laspy.ExtraBytesParams(name="true_label", type=np.uint8))
    if "predicted_label" not in dim_names:
        extras.append(laspy.ExtraBytesParams(name="predicted_label", type=np.uint8))
    if extras:
        header.add_extra_dims(extras)
    return header


def build_prediction_writer_header(input_header):
    header = copy.deepcopy(input_header)
    dim_names = [str(dim.name).lower() for dim in header.point_format.dimensions]
    if "predicted_label" not in dim_names:
        header.add_extra_dims([
            laspy.ExtraBytesParams(name="predicted_label", type=np.uint8),
        ])
    return header


def stream_write_subset_predictions(
    original_path: Path,
    subset_indices: np.ndarray,
    ids: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    out_path: Path,
    chunk_points: int = CHUNK_READ_POINTS,
) -> Path:
    subset_indices = np.asarray(subset_indices, dtype=np.int64)
    ids = np.asarray(ids, dtype=np.int64)
    y_true = np.asarray(y_true, dtype=np.int32)
    y_pred = np.asarray(y_pred, dtype=np.int32)

    order = np.argsort(subset_indices, kind="mergesort")
    subset_indices = subset_indices[order]
    ids = ids[order]
    y_true = y_true[order]
    y_pred = y_pred[order]

    with laspy.open(original_path) as reader:
        header = build_subset_prediction_writer_header(reader.header)
        with laspy.open(out_path, mode="w", header=header) as writer:
            start = 0
            for points in tqdm(reader.chunk_iterator(chunk_points), desc=f"write {out_path.stem}"):
                m = len(points)
                end = start + m

                lo = np.searchsorted(subset_indices, start, side="left")
                hi = np.searchsorted(subset_indices, end, side="left")

                if hi > lo:
                    local_idx = (subset_indices[lo:hi] - start).astype(np.int64, copy=False)
                    src_points = points[local_idx]
                    out_points = laspy.ScaleAwarePointRecord.zeros(len(local_idx), header=header)
                    out_points.copy_fields_from(src_points)
                    out_points["Id"] = ids[lo:hi].astype(np.int64, copy=False)
                    out_points["true_label"] = (y_true[lo:hi] + OUTPUT_CLASS_CODE_OFFSET).astype(np.uint8, copy=False)
                    out_points["predicted_label"] = (y_pred[lo:hi] + OUTPUT_CLASS_CODE_OFFSET).astype(np.uint8, copy=False)
                    out_points["classification"] = (y_pred[lo:hi] + OUTPUT_CLASS_CODE_OFFSET).astype(np.uint8, copy=False)
                    writer.write_points(out_points)

                start = end
    return out_path


def stream_write_predictions(original_path: Path, pred_labels_path: Path, out_path: Path, chunk_points: int = CHUNK_READ_POINTS) -> Path:
    pred_mm = np.memmap(pred_labels_path, dtype=np.uint8, mode="r")
    with laspy.open(original_path) as reader:
        header = build_prediction_writer_header(reader.header)
        with laspy.open(out_path, mode="w", header=header) as writer:
            start = 0
            for points in tqdm(reader.chunk_iterator(chunk_points), desc="write classified las"):
                m = len(points)
                chunk_pred = np.asarray(pred_mm[start:start + m], dtype=np.uint8)
                out_points = laspy.ScaleAwarePointRecord.zeros(m, header=header)
                out_points.copy_fields_from(points)
                out_points["classification"] = chunk_pred
                out_points["predicted_label"] = chunk_pred
                writer.write_points(out_points)
                start += m
    return out_path


def save_subset_prediction_outputs(model_name: str, meta_df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray):
    indices = meta_df["original_index"].to_numpy(np.int64)
    ids = meta_df["Id"].to_numpy(np.int64)
    stem = f"test_KitchenSink_{model_name}"

    las_path = TEST_OUTPUT_DIR / f"{stem}.las"
    cm_csv = REPORT_DIR / f"{stem}_confusion_matrix.csv"
    cm_png = FIG_DIR / f"{stem}_confusion_matrix.png"
    report_txt = REPORT_DIR / f"{stem}_classification_report.txt"
    per_class_csv = REPORT_DIR / f"{stem}_per_class_metrics.csv"
    top_png = FIG_DIR / f"{stem}_top.png"
    side_png = FIG_DIR / f"{stem}_side.png"

    labels_all = np.arange(len(CLASS_NAMES), dtype=np.int32)
    cm = confusion_matrix(y_true, y_pred, labels=labels_all)
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(cm_csv)
    save_confusion_matrix_png(cm, CLASS_NAMES, cm_png, title=f"Confusion Matrix: KitchenSink + {model_name}")

    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels_all,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    with open(report_txt, "w", encoding="utf-8") as f:
        f.write(report_text)

    precision, recall, f1_each, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels_all,
        zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class_name": CLASS_NAMES,
        "precision": precision,
        "recall": recall,
        "f1": f1_each,
        "support": support,
    })
    per_class_df.to_csv(per_class_csv, index=False)

    stream_write_subset_predictions(
        ORIGINAL_LAS_PATH,
        subset_indices=indices,
        ids=ids,
        y_true=y_true,
        y_pred=y_pred,
        out_path=las_path,
    )

    xyz = np.asarray(XYZ_MM[indices], dtype=np.float32)
    save_view_raster_png(xyz, y_pred, top_png, view="top")
    save_view_raster_png(xyz, y_pred, side_png, view="side")

    return {
        "test_las": str(las_path),
        "confusion_matrix_png": str(cm_png),
        "confusion_matrix_csv": str(cm_csv),
        "classification_report_txt": str(report_txt),
        "per_class_metrics_csv": str(per_class_csv),
        "top_view_png": str(top_png),
        "side_view_png": str(side_png),
        "confusion_matrix": cm,
        "y_pred": np.asarray(y_pred, dtype=np.int32),
    }


def rasterize_prediction_memmap_to_png(
    xyz_mm: np.memmap,
    pred_labels_path: Path,
    out_path: Path,
    view: str = "top",
    max_dim: int = VIEW_MAX_DIM,
    chunk_points: int = CHUNK_READ_POINTS,
    mode_filter_size: int = VIEW_MODE_FILTER_SIZE,
    mode_filter_passes: int = VIEW_MODE_FILTER_PASSES,
    background_rgb = VIEW_BACKGROUND,
    palette = None,
):
    n_points = int(xyz_mm.shape[0])
    pred_mm = np.memmap(pred_labels_path, dtype=np.uint8, mode="r")

    if view == "top":
        xa = xyz_mm[:, 0]
        ya = xyz_mm[:, 1]
    elif view == "side":
        xa = xyz_mm[:, 0]
        ya = xyz_mm[:, 2]
    else:
        raise ValueError(f"Unknown view: {view}")

    min_x, max_y, pixel_size, width, height = _estimate_pixel_size(xa, ya, max_dim=max_dim)
    counts = np.zeros((height * width, len(CLASS_NAMES)), dtype=np.uint32)

    for start in tqdm(range(0, n_points, chunk_points), desc=f"rasterize {view}"):
        end = min(start + chunk_points, n_points)
        xyz = np.asarray(xyz_mm[start:end], dtype=np.float32)
        labels = np.asarray(pred_mm[start:end], dtype=np.uint8).astype(np.int32, copy=False) - OUTPUT_CLASS_CODE_OFFSET

        if view == "top":
            xa_chunk = xyz[:, 0]
            ya_chunk = xyz[:, 1]
        else:
            xa_chunk = xyz[:, 0]
            ya_chunk = xyz[:, 2]

        valid_label = (labels >= 0) & (labels < len(CLASS_NAMES))
        if not np.any(valid_label):
            continue

        xa_chunk = xa_chunk[valid_label]
        ya_chunk = ya_chunk[valid_label]
        labels = labels[valid_label]

        col = np.floor((xa_chunk - min_x) / pixel_size).astype(np.int32)
        row = np.floor((max_y - ya_chunk) / pixel_size).astype(np.int32)

        mask = (row >= 0) & (row < height) & (col >= 0) & (col < width)
        if not np.any(mask):
            continue

        flat = row[mask].astype(np.int64) * width + col[mask].astype(np.int64)
        np.add.at(counts, (flat, labels[mask]), 1)

    majority = counts.argmax(axis=1).astype(np.uint8).reshape(height, width)
    occupancy = counts.sum(axis=1).reshape(height, width) > 0

    class_img = Image.fromarray(majority, mode="L")
    for _ in range(int(mode_filter_passes)):
        class_img = class_img.filter(ImageFilter.ModeFilter(size=int(mode_filter_size)))
    majority = np.array(class_img, dtype=np.uint8, copy=True)
    majority[~occupancy] = 255

    lut = build_palette_lut_from_labels(palette or CLASS_PALETTE, len(CLASS_NAMES))
    rgb = np.empty((height, width, 3), dtype=np.uint8)
    rgb[..., 0] = background_rgb[0]
    rgb[..., 1] = background_rgb[1]
    rgb[..., 2] = background_rgb[2]

    valid = majority != 255
    rgb[valid] = lut[majority[valid]]

    Image.fromarray(rgb, mode="RGB").save(out_path)
    return out_path



def predict_full_original_cloud_in_batches(model_bundle_path: Path, batch_size: int = FULL_PRED_BATCH_SIZE):
    bundle = joblib.load(model_bundle_path)
    model = bundle["model"]
    selected = list(bundle["selected_features"])
    context_params = bundle.get("context_params")
    cleanup_params = bundle.get("cleanup_params")
    n = int(MANIFEST["point_count"])

    pred_path = PRED_DIR / f"{ORIGINAL_LAS_PATH.stem}_{bundle['model_name']}_pred.uint8.mmap"

    if context_params is not None and hasattr(model, "predict_proba"):
        proba_path = PRED_DIR / f"{ORIGINAL_LAS_PATH.stem}_{bundle['model_name']}_proba.float32.mmap"
        proba_mm = np.memmap(proba_path, dtype=np.float32, mode="w+", shape=(n, len(CLASS_NAMES)))

        for start in tqdm(range(0, n, batch_size), desc="predict full cloud probabilities"):
            batch_idx = np.arange(start, min(start + batch_size, n), dtype=np.int64)
            batch_df = build_feature_table_for_indices(
                batch_idx,
                requested_columns=selected if FULL_PRED_BUILD_ONLY_SELECTED else None,
                verbose=False,
            )
            X_batch = batch_df.reindex(columns=selected, fill_value=np.nan)
            proba_mm[start:start + len(batch_idx)] = np.asarray(model.predict_proba(X_batch), dtype=np.float32)

        proba_mm.flush()

        pred_mm = np.memmap(pred_path, dtype=np.uint8, mode="w+", shape=(n,))
        coords_scaled = np.asarray(XYZ_MM, dtype=np.float32) * np.asarray(context_params["spatial_scales"], dtype=np.float32)[None, :]
        tree = cKDTree(coords_scaled)
        k = int(context_params["k"])
        alpha = float(context_params["alpha"])
        use_distance_weights = bool(context_params["use_distance_weights"])

        for start in tqdm(range(0, n, batch_size), desc="context smooth full cloud"):
            stop = min(start + batch_size, n)
            dists, idx = tree.query(coords_scaled[start:stop], k=min(n, k + 1), workers=-1)
            if idx.ndim == 1:
                dists = dists[:, None]
                idx = idx[:, None]
            if idx.shape[1] > 1:
                dists = np.asarray(dists[:, 1:], dtype=np.float32)
                idx = np.asarray(idx[:, 1:], dtype=np.int64)
                neigh_proba = np.asarray(proba_mm[idx], dtype=np.float32)
                if use_distance_weights:
                    weights = 1.0 / np.maximum(dists, 1e-3)
                    weights = weights / np.maximum(weights.sum(axis=1, keepdims=True), 1e-6)
                    neigh_mean = (neigh_proba * weights[..., None]).sum(axis=1).astype(np.float32)
                else:
                    neigh_mean = neigh_proba.mean(axis=1).astype(np.float32)
                self_proba = np.asarray(proba_mm[start:stop], dtype=np.float32)
                smoothed = ((1.0 - alpha) * self_proba + alpha * neigh_mean).astype(np.float32)
                smoothed = smoothed / np.clip(smoothed.sum(axis=1, keepdims=True), 1e-12, None)
                pred_mm[start:stop] = np.argmax(smoothed, axis=1).astype(np.uint8) + OUTPUT_CLASS_CODE_OFFSET
            else:
                pred_mm[start:stop] = np.argmax(np.asarray(proba_mm[start:stop], dtype=np.float32), axis=1).astype(np.uint8) + OUTPUT_CLASS_CODE_OFFSET

        pred_mm.flush()
    else:
        pred_mm = np.memmap(pred_path, dtype=np.uint8, mode="w+", shape=(n,))
        for start in tqdm(range(0, n, batch_size), desc="predict full cloud"):
            batch_idx = np.arange(start, min(start + batch_size, n), dtype=np.int64)
            batch_df = build_feature_table_for_indices(
                batch_idx,
                requested_columns=selected if FULL_PRED_BUILD_ONLY_SELECTED else None,
                verbose=False,
            )
            X_batch = batch_df.reindex(columns=selected, fill_value=np.nan)
            pred_batch = np.asarray(model.predict(X_batch), dtype=np.uint8)
            pred_mm[start:start + len(batch_idx)] = pred_batch + OUTPUT_CLASS_CODE_OFFSET
        pred_mm.flush()

    if cleanup_params is not None:
        print("Applying component cleanup to full-cloud predictions...")
        pred_mm = np.memmap(pred_path, dtype=np.uint8, mode="r+", shape=(n,))
        pred_zero_based = np.asarray(pred_mm, dtype=np.int32) - OUTPUT_CLASS_CODE_OFFSET
        xyz_df = pd.DataFrame({
            "x": np.asarray(XYZ_MM[:, 0], dtype=np.float32),
            "y": np.asarray(XYZ_MM[:, 1], dtype=np.float32),
        })
        pred_clean = cleanup_point_predictions_by_components(
            xyz_df,
            pred_zero_based,
            grid_size=float(cleanup_params["grid_size"]),
            min_cells_by_class=dict(cleanup_params["min_cells_by_class"]),
            connectivity=int(cleanup_params["connectivity"]),
        )
        pred_mm[:] = pred_clean.astype(np.uint8) + OUTPUT_CLASS_CODE_OFFSET
        pred_mm.flush()

    out_path = PRED_DIR / f"{ORIGINAL_LAS_PATH.stem}_classified_{bundle['model_name']}.las"
    out_path = stream_write_predictions(ORIGINAL_LAS_PATH, pred_path, out_path)
    return out_path, pred_path


## Choose the model

This section selects the single model that will be treated as the notebook's main output. By default, the ranking rule is:

1. highest holdout **macro F1**
2. highest **spatial cross-validation macro F1**
3. highest ordinary **cross-validation macro F1**
4. lower prediction time
5. lower training time

This ranking rule tries to favor the model that is strongest on the held-out test set while still breaking close ties in favor of better spatial generalization and lower runtime.

The selected model is then written to disk with `joblib`. The saved bundle contains the fitted model plus the exact feature list, class names, and neighborhood settings needed to reproduce predictions later. `joblib` is commonly used for persisting Python objects that include large NumPy arrays, which makes it a practical choice for fitted scikit-learn style pipelines.

**References**

- [scikit-learn: model persistence](https://scikit-learn.org/stable/model_persistence.html)
- [joblib persistence](https://joblib.readthedocs.io/en/latest/persistence.html)
- [joblib.dump](https://joblib.readthedocs.io/en/latest/generated/joblib.dump.html)


In [ ]:

all_results = []




for row in cv_results_df.to_dict(orient="records"):
    model_name = str(row["model"])
    trained_entry = trained_models[model_name]
    model = trained_entry["model"]
    context_params = trained_entry.get("context_params")
    cleanup_params = trained_entry.get("cleanup_params")

    result = dict(row)
    if have_test:
        y_pred = predict_with_context_postprocess(model, X_test_sel, context_params=context_params).astype(np.int32, copy=False)
        if cleanup_params is not None:
            y_pred = cleanup_point_predictions_by_components(
                X_test_sel,
                y_pred,
                grid_size=float(cleanup_params["grid_size"]),
                min_cells_by_class=dict(cleanup_params["min_cells_by_class"]),
                connectivity=int(cleanup_params["connectivity"]),
            )

        outputs = save_subset_prediction_outputs(
            model_name=model_name,
            meta_df=TEST_META,
            y_true=y_test,
            y_pred=y_pred,
        )
        result.update({
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "kappa": float(cohen_kappa_score(y_test, y_pred)),
            "f1_macro": float(f1_score(y_test, y_pred, average="macro")),
            "weighted_f1": float(f1_score(y_test, y_pred, average="weighted")),
            "test_las": outputs["test_las"],
            "confusion_matrix_png": outputs["confusion_matrix_png"],
            "confusion_matrix_csv": outputs["confusion_matrix_csv"],
            "classification_report_txt": outputs["classification_report_txt"],
            "per_class_metrics_csv": outputs["per_class_metrics_csv"],
            "top_view_png": outputs["top_view_png"],
            "side_view_png": outputs["side_view_png"],
            "confusion_matrix": outputs["confusion_matrix"],
            "y_pred": outputs["y_pred"],
        })
    else:
        result.update({
            "test_las": None,
            "confusion_matrix_png": None,
            "confusion_matrix_csv": None,
            "classification_report_txt": None,
            "per_class_metrics_csv": None,
            "top_view_png": None,
            "side_view_png": None,
            "confusion_matrix": None,
            "y_pred": None,
        })

    result["pipeline"] = model
    result["context_params"] = context_params
    result["cleanup_params"] = cleanup_params
    result["used_features"] = list(selected_features)
    all_results.append(result)

RESULTS_DF = pd.DataFrame(
    [
        {
            k: v
            for k, v in row.items()
            if k not in {"pipeline", "y_pred", "confusion_matrix", "used_features", "selected_features", "context_params", "cleanup_params"}
        }
        for row in all_results
    ]
).sort_values(
    ["f1_macro", "spatial_cv_macro_f1_mean", "cv_macro_f1_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

RESULTS_DF.to_csv(REPORT_DIR / "model_results_full.csv", index=False)
display(RESULTS_DF)

comparison_table = RESULTS_DF[[
    "strategy",
    "model",
    "fit_rows",
    "selected_feature_count",
    "cv_macro_f1_mean",
    "cv_macro_f1_std",
    "spatial_cv_macro_f1_mean",
    "spatial_cv_macro_f1_std",
    "train_time_sec",
    "test_time_sec",
    "accuracy",
    "kappa",
    "f1_macro",
    "weighted_f1",
    "calibration_macro_f1",
    "calibration_notes",
    "context_enabled",
    "component_cleanup_enabled",
    "test_las",
    "confusion_matrix_png",
    "top_view_png",
    "side_view_png",
]].copy()
comparison_table.to_csv(REPORT_DIR / "comparison_table.csv", index=False)
display(comparison_table)

for result in all_results:
    print(f"{result['strategy']} + {result['model']}")
    print(f"Selected features ({result['selected_feature_count']}):")
    print(result["used_features"])
    if result["context_params"] is not None:
        print("Context smoothing:", result["context_params"])
    if result["cleanup_params"] is not None:
        print("Component cleanup:", result["cleanup_params"])
    if result["confusion_matrix"] is not None:
        print("Confusion matrix:")
        display(pd.DataFrame(result["confusion_matrix"], index=CLASS_NAMES, columns=CLASS_NAMES))
    else:
        print("No holdout test split available.")

BEST_OVERRIDE = None

if BEST_OVERRIDE is None:
    best_result = sorted(
        all_results,
        key=lambda row: (
            -(row["f1_macro"] if pd.notna(row["f1_macro"]) else -np.inf),
            -(row["spatial_cv_macro_f1_mean"] if pd.notna(row["spatial_cv_macro_f1_mean"]) else -np.inf),
            -row["cv_macro_f1_mean"],
            row["test_time_sec"] if pd.notna(row["test_time_sec"]) else np.inf,
            row["train_time_sec"],
        ),
    )[0]
else:
    best_model_name = str(BEST_OVERRIDE)
    best_result = next(row for row in all_results if row["model"] == best_model_name)

print("Selected best model:")
print({
    "strategy": best_result["strategy"],
    "model": best_result["model"],
    "f1_macro": best_result["f1_macro"],
    "spatial_cv_macro_f1_mean": best_result["spatial_cv_macro_f1_mean"],
    "kappa": best_result["kappa"],
    "accuracy": best_result["accuracy"],
    "context_params": best_result["context_params"],
    "cleanup_params": best_result["cleanup_params"],
})

joblib.dump(best_result["pipeline"], MODEL_DIR / "best_test_pipeline.joblib")

bundle = {
    "model_name": best_result["model"],
    "model": best_result["pipeline"],
    "context_params": best_result["context_params"],
    "cleanup_params": best_result["cleanup_params"],
    "selected_features": list(best_result["used_features"]),
    "grid_sizes": GRID_SIZES,
    "support_grid_size": SUPPORT_GRID_SIZE,
    "support_ks": SUPPORT_KS,
    "support_geom_ks": SUPPORT_GEOM_KS,
    "support_adaptive_geom_ks": SUPPORT_ADAPTIVE_GEOM_KS,
    "support_terrain_ks": SUPPORT_TERRAIN_KS,
    "class_names": CLASS_NAMES,
}
joblib.dump(bundle, MODEL_DIR / "best_model_bundle.joblib")
joblib.dump(best_result["pipeline"], MODEL_DIR / "best_full_pipeline.joblib")

FULL_OUTPUT_SUMMARY = None
if RUN_FULL_CLOUD_PREDICTION:
    full_las_path, full_pred_path = predict_full_original_cloud_in_batches(
        MODEL_DIR / "best_model_bundle.joblib",
        batch_size=FULL_PRED_BATCH_SIZE,
    )

    full_top_png = FIG_DIR / f"full_KitchenSinkGroundSpatialCleanup_{best_result['model']}_top.png"
    full_side_png = FIG_DIR / f"full_KitchenSinkGroundSpatialCleanup_{best_result['model']}_side.png"

    rasterize_prediction_memmap_to_png(XYZ_MM, full_pred_path, full_top_png, view="top")
    rasterize_prediction_memmap_to_png(XYZ_MM, full_pred_path, full_side_png, view="side")

    FULL_OUTPUT_SUMMARY = {
        "full_las_path": str(full_las_path),
        "full_pred_memmap": str(full_pred_path),
        "full_top_png": str(full_top_png),
        "full_side_png": str(full_side_png),
    }
    print("Full cloud outputs:")
    print(FULL_OUTPUT_SUMMARY)


write test_KitchenSink_HierarchicalExtraTreesTunedContextKNN: 19it [00:00, 32.80it/s]
write test_KitchenSink_ExtraTreesRegularizedContextKNN: 19it [00:00, 54.05it/s]
write test_KitchenSink_ExtraTreesSoftVoteTunedContextKNN: 19it [00:00, 54.99it/s]
write test_KitchenSink_ExtraTreesSoftVoteTuned: 19it [00:00, 56.02it/s]
write test_KitchenSink_ExtraTreesRegularized: 19it [00:00, 56.33it/s]
write test_KitchenSink_ExtraTreesConservativeContextKNN: 19it [00:00, 51.57it/s]
write test_KitchenSink_ExtraTreesConservative: 19it [00:00, 56.02it/s]
write test_KitchenSink_ExtraTreesContextKNN: 19it [00:00, 52.95it/s]
write test_KitchenSink_ExtraTrees: 19it [00:00, 55.51it/s]
write test_KitchenSink_HierarchicalExtraTreesTunedContextKNNComponentClean: 19it [00:00, 51.86it/s]
write test_KitchenSink_ExtraTreesSoftVoteTunedComponentClean: 19it [00:00, 55.17it/s]
write test_KitchenSink_ExtraTreesSoftVoteTunedContextKNNComponentClean: 19it [00:00, 51.85it/s]
write test_KitchenSink_ExtraTreesRegularizedCont

,strategy,model,selected_feature_count,fit_rows,cv_macro_f1_mean,cv_macro_f1_std,spatial_cv_macro_f1_mean,spatial_cv_macro_f1_std,train_time_sec,test_time_sec,...,calibration_notes,context_enabled,component_cleanup_enabled,test_las,confusion_matrix_png,confusion_matrix_csv,classification_report_txt,per_class_metrics_csv,top_view_png,side_view_png
0,KitchenSinkGroundSpatialCleanup,HierarchicalExtraTreesTunedContextKNN,64,148092,0.954975,0.002757,0.890413,0.021384,390.346248,2.800854,...,"{'k': 16, 'alpha': 0.2, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
1,KitchenSinkGroundSpatialCleanup,ExtraTreesRegularizedContextKNN,64,148092,0.956988,0.002866,0.888413,0.023387,18.896895,1.881294,...,"{'k': 32, 'alpha': 0.2, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
2,KitchenSinkGroundSpatialCleanup,ExtraTreesSoftVoteTunedContextKNN,64,148092,0.960648,0.002824,0.896670,0.024108,241.585155,4.536458,...,"{'k': 8, 'alpha': 0.2, 'spatial_columns': ('x'...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
3,KitchenSinkGroundSpatialCleanup,ExtraTreesSoftVoteTuned,64,148092,0.960648,0.002824,0.896670,0.024108,241.585155,4.410110,...,"(1.05, 0.95, 1.0, 1.2)",False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
4,KitchenSinkGroundSpatialCleanup,ExtraTreesRegularized,64,148092,0.956988,0.002866,0.888413,0.023387,18.896895,1.493801,...,,False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
5,KitchenSinkGroundSpatialCleanup,ExtraTreesConservativeContextKNN,64,148092,0.953731,0.002037,0.897339,0.021557,33.295453,1.768863,...,"{'k': 8, 'alpha': 0.35, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
6,KitchenSinkGroundSpatialCleanup,ExtraTreesConservative,64,148092,0.953731,0.002037,0.897339,0.021557,33.295453,1.644556,...,,False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,output

,strategy,model,fit_rows,selected_feature_count,cv_macro_f1_mean,cv_macro_f1_std,spatial_cv_macro_f1_mean,spatial_cv_macro_f1_std,train_time_sec,test_time_sec,...,f1_macro,weighted_f1,calibration_macro_f1,calibration_notes,context_enabled,component_cleanup_enabled,test_las,confusion_matrix_png,top_view_png,side_view_png
0,KitchenSinkGroundSpatialCleanup,HierarchicalExtraTreesTunedContextKNN,148092,64,0.954975,0.002757,0.890413,0.021384,390.346248,2.800854,...,0.785558,0.929843,0.943969,"{'k': 16, 'alpha': 0.2, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
1,KitchenSinkGroundSpatialCleanup,ExtraTreesRegularizedContextKNN,148092,64,0.956988,0.002866,0.888413,0.023387,18.896895,1.881294,...,0.773171,0.920882,0.957662,"{'k': 32, 'alpha': 0.2, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
2,KitchenSinkGroundSpatialCleanup,ExtraTreesSoftVoteTunedContextKNN,148092,64,0.960648,0.002824,0.896670,0.024108,241.585155,4.536458,...,0.770578,0.918210,0.963977,"{'k': 8, 'alpha': 0.2, 'spatial_columns': ('x'...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
3,KitchenSinkGroundSpatialCleanup,ExtraTreesSoftVoteTuned,148092,64,0.960648,0.002824,0.896670,0.024108,241.585155,4.410110,...,0.770436,0.916991,0.964281,"(1.05, 0.95, 1.0, 1.2)",False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
4,KitchenSinkGroundSpatialCleanup,ExtraTreesRegularized,148092,64,0.956988,0.002866,0.888413,0.023387,18.896895,1.493801,...,0.769765,0.918852,NaN,,False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
5,KitchenSinkGroundSpatialCleanup,ExtraTreesConservativeContextKNN,148092,64,0.953731,0.002037,0.897339,0.021557,33.295453,1.768863,...,0.764564,0.918212,0.956394,"{'k': 8, 'alpha': 0.35, 'spatial_columns': ('x...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
6,KitchenSinkGroundSpatialCleanup,ExtraTreesConservative,148092,64,0.953731,0.002037,0.897339,0.021557,33.295453,1.644556,...,0.758983,0.914822,NaN,,False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
7,KitchenSinkGroundSpatialCleanup,ExtraTreesContextKNN,148092,64,0.962860,0.002996,0.879985,0.017607,8.889677,1.495483,...,0.758194,0.919204,0.960486,"{'k': 8, 'alpha': 0.2, 'spatial_columns': ('x'...",True,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
8,KitchenSinkGroundSpatialCleanup,ExtraTrees,148092,64,0.962860,0.002996,0.879985,0.017607,8.889677,1.232653,...,0.756047,0.917463,NaN,,False,False,outputs_multimodel_kitchensink_learned\test_pr...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
9,KitchenSinkGroundSpatialCleanup,HierarchicalExtraTreesTunedContextKNNComponent...,148092,64

KitchenSinkGroundSpatialCleanup + HierarchicalExtraTreesTunedContextKNN
Selected features (64):
['x', 'y', 'z', 'hag_terrain_xy_q10_k32', 'terrain_xy_relief_k32', 'geom_planarity_k16', 'geom_planarity_k32', 'geom_sphericity_k16', 'geom_linearity_k16', 'geom_linearity_k32', 'geom_eigenentropy_k16', 'geom_eigenentropy_k32', 'geom_normal_z_abs_k16', 'geom_normal_z_abs_k32', 'is_first_return', 'return_fraction', 'return_number', 'geom_adapt_planarity', 'geom_adapt_sphericity', 'geom_adapt_omnivariance', 'geom_adapt_eigenentropy', 'geom_adapt_normal_z_abs', 'geom_adapt_z_std', 'geom_adapt_z_range', 'ground_proxy_z', 'hag_ground_proxy', 'ground_proxy_residual', 'ground_proxy_slope_mag', 'ground_proxy_curvature', 'hag_ground_proxy_minus_terrain_q10_k32', 'ground_proxy_slope_planarity_ratio', 'hag_terrain_xy_min_k16', 'hag_min_g16.0', 'hag_min_g8.0', 'hag_terrain_xy_q10_k16', 'nbr_ground_relief_g8.0', 'nbr_zrange_mean_g8.0', 'ground_slope_mag_g8.0', 'terrain_xy_relief_k16', 'nbr_ground_relief_

,Buildings,Trees,Ground,Cars
Buildings,27732,5278,13,0
Trees,2758,28416,237,187
Ground,2247,1249,107409,225
Cars,27,352,37,364


...

KitchenSinkGroundSpatialCleanup + RandomForestComponentClean
Selected features (64):
['x', 'y', 'z', 'hag_terrain_xy_q10_k32', 'terrain_xy_relief_k32', 'geom_planarity_k16', 'geom_planarity_k32', 'geom_sphericity_k16', 'geom_linearity_k16', 'geom_linearity_k32', 'geom_eigenentropy_k16', 'geom_eigenentropy_k32', 'geom_normal_z_abs_k16', 'geom_normal_z_abs_k32', 'is_first_return', 'return_fraction', 'return_number', 'geom_adapt_planarity', 'geom_adapt_sphericity', 'geom_adapt_omnivariance', 'geom_adapt_eigenentropy', 'geom_adapt_normal_z_abs', 'geom_adapt_z_std', 'geom_adapt_z_range', 'ground_proxy_z', 'hag_ground_proxy', 'ground_proxy_residual', 'ground_proxy_slope_mag', 'ground_proxy_curvature', 'hag_ground_proxy_minus_terrain_q10_k32', 'ground_proxy_slope_planarity_ratio', 'hag_terrain_xy_min_k16', 'hag_min_g16.0', 'hag_min_g8.0', 'hag_terrain_xy_q10_k16', 'nbr_ground_relief_g8.0', 'nbr_zrange_mean_g8.0', 'ground_slope_mag_g8.0', 'terrain_xy_relief_k16', 'nbr_ground_relief_g4.0', 'nbr

,Buildings,Trees,Ground,Cars
Buildings,15934,16892,197,0
Trees,431,30550,616,1
Ground,1177,8239,101692,22
Cars,0,723,47,10


Selected best model:
{'strategy': 'KitchenSinkGroundSpatialCleanup', 'model': 'HierarchicalExtraTreesTunedContextKNN', 'f1_macro': 0.7855576311263639, 'spatial_cv_macro_f1_mean': 0.8904127506313905, 'kappa': 0.8690333105754539, 'accuracy': 0.9285677869609303, 'context_params': {'k': 16, 'alpha': 0.2, 'spatial_columns': ('x', 'y', 'z'), 'spatial_scales': (1.0, 1.0, 2.0), 'use_distance_weights': True, 'calibration_macro_f1': 0.9439691672770654}, 'cleanup_params': None}


context smooth full cloud: 100%|██████████| 378/378 [00:25<00:00, 14.76it/s]
write classified las: 19it [00:05,  3.74it/s]
rasterize side: 100%|██████████| 19/19 [00:01<00:00, 10.93it/s]

Full cloud outputs:
{'full_las_path': 'outputs_multimodel_kitchensink_learned\\predictions\\original_point_cloud_classified_HierarchicalExtraTreesTunedContextKNN.las', 'full_pred_memmap': 'outputs_multimodel_kitchensink_learned\\predictions\\original_point_cloud_HierarchicalExtraTreesTunedContextKNN_pred.uint8.mmap', 'full_top_png': 'outputs_multimodel_kitchensink_learned\\reports\\figures\\full_KitchenSinkGroundSpatialCleanup_HierarchicalExtraTreesTunedContextKNN_top.png', 'full_side_png': 'outputs_multimodel_kitchensink_learned\\reports\\figures\\full_KitchenSinkGroundSpatialCleanup_HierarchicalExtraTreesTunedContextKNN_side.png'}


In [ ]:
EXTRA_FULL_PRED_MODEL_NAMES = [
    "XGBoostContextKNN",
    "XGBoost",
]

EXTRA_FULL_OUTPUT_SUMMARIES = []

for extra_model_name in EXTRA_FULL_PRED_MODEL_NAMES:
    matching_results = [row for row in all_results if str(row["model"]) == str(extra_model_name)]
    if not matching_results:
        print(f"Skipping full-cloud prediction for {extra_model_name}: model not found in all_results.")
        continue

    extra_result = sorted(
        matching_results,
        key=lambda row: (
            -(row["f1_macro"] if pd.notna(row["f1_macro"]) else -np.inf),
            -(row["spatial_cv_macro_f1_mean"] if pd.notna(row["spatial_cv_macro_f1_mean"]) else -np.inf),
            -row["cv_macro_f1_mean"],
            row["test_time_sec"] if pd.notna(row["test_time_sec"]) else np.inf,
            row["train_time_sec"] if pd.notna(row["train_time_sec"]) else np.inf,
        ),
    )[0]

    extra_bundle = {
        "model_name": extra_result["model"],
        "model": extra_result["pipeline"],
        "context_params": extra_result["context_params"],
        "cleanup_params": extra_result["cleanup_params"],
        "selected_features": list(extra_result["used_features"]),
        "grid_sizes": GRID_SIZES,
        "support_grid_size": SUPPORT_GRID_SIZE,
        "support_ks": SUPPORT_KS,
        "support_geom_ks": SUPPORT_GEOM_KS,
        "support_adaptive_geom_ks": SUPPORT_ADAPTIVE_GEOM_KS,
        "support_terrain_ks": SUPPORT_TERRAIN_KS,
        "class_names": CLASS_NAMES,
    }

    extra_bundle_path = MODEL_DIR / f"{extra_result['model']}_full_model_bundle.joblib"
    joblib.dump(extra_bundle, extra_bundle_path)
    joblib.dump(extra_result["pipeline"], MODEL_DIR / f"{extra_result['model']}_full_pipeline.joblib")

    extra_full_las_path, extra_full_pred_path = predict_full_original_cloud_in_batches(
        extra_bundle_path,
        batch_size=FULL_PRED_BATCH_SIZE,
    )

    extra_full_top_png = FIG_DIR / f"full_{extra_result['strategy']}_{extra_result['model']}_top.png"
    extra_full_side_png = FIG_DIR / f"full_{extra_result['strategy']}_{extra_result['model']}_side.png"

    rasterize_prediction_memmap_to_png(XYZ_MM, extra_full_pred_path, extra_full_top_png, view="top")
    rasterize_prediction_memmap_to_png(XYZ_MM, extra_full_pred_path, extra_full_side_png, view="side")

    extra_summary = {
        "strategy": extra_result["strategy"],
        "model": extra_result["model"],
        "context_enabled": bool(extra_result["context_params"] is not None),
        "component_cleanup_enabled": bool(extra_result["cleanup_params"] is not None),
        "full_las_path": str(extra_full_las_path),
        "full_pred_memmap": str(extra_full_pred_path),
        "full_top_png": str(extra_full_top_png),
        "full_side_png": str(extra_full_side_png),
        "bundle_path": str(extra_bundle_path),
    }
    EXTRA_FULL_OUTPUT_SUMMARIES.append(extra_summary)

EXTRA_FULL_OUTPUTS_DF = pd.DataFrame(EXTRA_FULL_OUTPUT_SUMMARIES)
if len(EXTRA_FULL_OUTPUTS_DF):
    EXTRA_FULL_OUTPUTS_DF.to_csv(REPORT_DIR / "extra_full_cloud_outputs.csv", index=False)

print("Extra full-cloud outputs:")
display(EXTRA_FULL_OUTPUTS_DF)

context smooth full cloud: 100%|██████████| 378/378 [00:21<00:00, 17.54it/s]
write classified las: 19it [00:06,  2.77it/s]
predict full cloud: 100%|██████████| 378/378 [22:33<00:00,  3.58s/it]
write classified las: 19it [00:05,  3.37it/s]
rasterize side: 100%|██████████| 19/19 [00:01<00:00, 10.24it/s]


Extra full-cloud outputs:


,strategy,model,context_enabled,component_cleanup_enabled,full_las_path,full_pred_memmap,full_top_png,full_side_png,bundle_path
0,KitchenSinkGroundSpatialCleanup,XGBoostContextKNN,True,False,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\models\...
1,KitchenSinkGroundSpatialCleanup,XGBoost,False,False,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\models\...


## Higher-detail preview rendering for the full-cloud outputs

The standard preview images are useful for quick checking, but they flatten the scene into large color patches. This section re-renders the saved full-cloud prediction arrays at higher spatial detail and varies brightness using a per-pixel height surface. The class colors stay the same, but roofs, tree crowns, and street-level structure read more clearly.

This is still a diagnostic view, not a replacement for opening the full LAS in a point-cloud viewer. The goal is to preserve the class map while making local relief easier to see.

In [ ]:
from scipy.ndimage import gaussian_filter

DETAILED_VIEW_MAX_DIM = 3200
DETAILED_PIXEL_SIZE_SCALE = 2
DETAILED_MODE_FILTER_SIZE = 3
DETAILED_MODE_FILTER_PASSES = 0
DETAILED_SHADE_SIGMA = 1.25
DETAILED_HEIGHT_STRENGTH = 0.20
DETAILED_RELIEF_STRENGTH = 0.22
DETAILED_DENSITY_STRENGTH = 0.08
DETAILED_EDGE_DARKEN = 0.92


def _normalize_on_mask(arr, mask, q_low=2.0, q_high=98.0):
    """
    Normalize values to 0..1 using only occupied pixels.
    Robust percentile limits keep a few extreme values from dominating the shading.
    """
    out = np.zeros(arr.shape, dtype=np.float32)
    if not np.any(mask):
        return out

    vals = np.asarray(arr[mask], dtype=np.float32)
    lo = float(np.nanpercentile(vals, q_low))
    hi = float(np.nanpercentile(vals, q_high))

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo + 1e-6:
        out[mask] = 0.5
        return out

    out[mask] = np.clip((arr[mask] - lo) / (hi - lo), 0.0, 1.0)
    return out


def _fill_missing_by_blur(arr, mask, sigma=2.0):
    """
    Fill empty pixels from nearby occupied pixels so the shading surface stays continuous.
    The class map is not changed here, only the shading surface used for visualization.
    """
    arr = np.asarray(arr, dtype=np.float32)
    mask_f = np.asarray(mask, dtype=np.float32)

    weighted_sum = gaussian_filter(np.where(mask, arr, 0.0), sigma=sigma, mode="nearest")
    weight = gaussian_filter(mask_f, sigma=sigma, mode="nearest")

    out = arr.copy()
    fill_mask = (~mask) & (weight > 1e-6)
    out[fill_mask] = weighted_sum[fill_mask] / weight[fill_mask]
    return out


def rasterize_prediction_memmap_to_png_detailed(
    xyz_mm: np.memmap,
    pred_labels_path: Path,
    out_path: Path,
    view: str = "top",
    max_dim: int = DETAILED_VIEW_MAX_DIM,
    pixel_size_scale: float = DETAILED_PIXEL_SIZE_SCALE,
    chunk_points: int = CHUNK_READ_POINTS,
    mode_filter_size: int = DETAILED_MODE_FILTER_SIZE,
    mode_filter_passes: int = DETAILED_MODE_FILTER_PASSES,
    shade_sigma: float = DETAILED_SHADE_SIGMA,
    height_strength: float = DETAILED_HEIGHT_STRENGTH,
    relief_strength: float = DETAILED_RELIEF_STRENGTH,
    density_strength: float = DETAILED_DENSITY_STRENGTH,
    edge_darken: float = DETAILED_EDGE_DARKEN,
    background_rgb=VIEW_BACKGROUND,
    palette=None,
):
    """
    Render a prediction memmap as a PNG with:
    - one dominant class per output pixel
    - optional light mode filtering to connect tiny breaks
    - height and local relief shading for depth
    - slight edge darkening so class boundaries read more clearly
    """
    n_points = int(xyz_mm.shape[0])
    pred_mm = np.memmap(pred_labels_path, dtype=np.uint8, mode="r")

    if view == "top":
        xa = xyz_mm[:, 0]
        ya = xyz_mm[:, 1]
    elif view == "side":
        xa = xyz_mm[:, 0]
        ya = xyz_mm[:, 2]
    else:
        raise ValueError(f"Unknown view: {view}")

    min_x, max_y, pixel_size, _, _ = _estimate_pixel_size(xa, ya, max_dim=max_dim)

    # Slightly larger pixels help the view connect without making it overly soft.
    pixel_size = float(pixel_size) * float(pixel_size_scale)

    min_y = float(np.min(ya))
    max_x = float(np.max(xa))
    width = max(1, int(np.ceil((max_x - float(min_x)) / pixel_size)) + 1)
    height = max(1, int(np.ceil((float(max_y) - min_y) / pixel_size)) + 1)

    n_classes = len(CLASS_NAMES)
    flat_size = height * width

    # Per-pixel class vote counts.
    counts = np.zeros((flat_size, n_classes), dtype=np.uint32)

    # Number of original points that landed in each output pixel.
    point_density = np.zeros(flat_size, dtype=np.uint32)

    # Surfaces used for shading.
    scalar_sum = np.zeros(flat_size, dtype=np.float64)
    scalar_count = np.zeros(flat_size, dtype=np.uint32)
    scalar_max = np.full(flat_size, -np.inf, dtype=np.float32)

    for start in tqdm(range(0, n_points, chunk_points), desc=f"render detailed {view}"):
        end = min(start + chunk_points, n_points)

        xyz = np.asarray(xyz_mm[start:end], dtype=np.float32)
        labels = np.asarray(pred_mm[start:end], dtype=np.uint8).astype(np.int32, copy=False) - OUTPUT_CLASS_CODE_OFFSET

        if view == "top":
            xa_chunk = xyz[:, 0]
            ya_chunk = xyz[:, 1]
            scalar_chunk = xyz[:, 2]
        else:
            xa_chunk = xyz[:, 0]
            ya_chunk = xyz[:, 2]
            scalar_chunk = xyz[:, 2]

        valid_label = (labels >= 0) & (labels < n_classes)
        if not np.any(valid_label):
            continue

        xa_chunk = xa_chunk[valid_label]
        ya_chunk = ya_chunk[valid_label]
        scalar_chunk = scalar_chunk[valid_label]
        labels = labels[valid_label]

        col = np.floor((xa_chunk - float(min_x)) / pixel_size).astype(np.int32)
        row = np.floor((float(max_y) - ya_chunk) / pixel_size).astype(np.int32)

        in_bounds = (row >= 0) & (row < height) & (col >= 0) & (col < width)
        if not np.any(in_bounds):
            continue

        flat = row[in_bounds].astype(np.int64) * width + col[in_bounds].astype(np.int64)
        labels_m = labels[in_bounds]
        scalar_m = scalar_chunk[in_bounds]

        np.add.at(counts, (flat, labels_m), 1)
        np.add.at(point_density, flat, 1)
        np.add.at(scalar_sum, flat, scalar_m)
        np.add.at(scalar_count, flat, 1)
        np.maximum.at(scalar_max, flat, scalar_m)

    majority = counts.argmax(axis=1).astype(np.uint8).reshape(height, width)
    occupancy = point_density.reshape(height, width) > 0

    # Optional mode filtering can merge tiny isolated breaks in the class image.
    if mode_filter_passes > 0:
        class_img = Image.fromarray(majority, mode="L")
        for _ in range(int(mode_filter_passes)):
            class_img = class_img.filter(ImageFilter.ModeFilter(size=int(mode_filter_size)))
        majority = np.array(class_img, dtype=np.uint8, copy=True)

    # Mark empty pixels so they keep the background color.
    majority[~occupancy] = 255

    mean_scalar = np.divide(
        scalar_sum,
        np.maximum(scalar_count, 1),
        out=np.zeros_like(scalar_sum, dtype=np.float64),
        where=scalar_count > 0,
    ).astype(np.float32)

    top_scalar = scalar_max.copy()
    top_scalar[~np.isfinite(top_scalar)] = 0.0

    # Top view uses the highest point in each pixel so crowns and roof tops read better.
    # Side view uses the average height in each pixel so vertical structure reads more smoothly.
    if view == "top":
        scalar_map = top_scalar.reshape(height, width)
    else:
        scalar_map = mean_scalar.reshape(height, width)

    scalar_map = _fill_missing_by_blur(scalar_map, occupancy, sigma=max(1.0, shade_sigma * 1.5))
    scalar_smooth = gaussian_filter(scalar_map, sigma=shade_sigma, mode="nearest")

    # Local slope-like shading from the smoothed height surface.
    grad_y, grad_x = np.gradient(scalar_smooth)
    relief = (-0.7 * grad_x + 0.7 * grad_y).astype(np.float32)

    height_norm = _normalize_on_mask(scalar_smooth, occupancy)
    relief_norm = _normalize_on_mask(relief, occupancy)
    density_norm = _normalize_on_mask(np.log1p(point_density.reshape(height, width).astype(np.float32)), occupancy)

    shade = np.ones((height, width), dtype=np.float32)
    shade[occupancy] = (
        0.78
        + height_strength * height_norm[occupancy]
        + relief_strength * (relief_norm[occupancy] - 0.5)
        + density_strength * (density_norm[occupancy] - 0.5)
    )
    shade = np.clip(shade, 0.55, 1.18)

    lut = build_palette_lut_from_labels(palette or CLASS_PALETTE, n_classes)

    rgb = np.empty((height, width, 3), dtype=np.uint8)
    rgb[..., 0] = background_rgb[0]
    rgb[..., 1] = background_rgb[1]
    rgb[..., 2] = background_rgb[2]

    valid = majority != 255
    rgb[valid] = lut[majority[valid]]

    rgb_f = rgb.astype(np.float32)
    rgb_f[valid] *= shade[valid][..., None]

    # Slight edge darkening helps separate neighboring classes without changing the class colors.
    edge_mask = np.zeros((height, width), dtype=bool)
    edge_mask[1:, :] |= majority[1:, :] != majority[:-1, :]
    edge_mask[:-1, :] |= majority[:-1, :] != majority[1:, :]
    edge_mask[:, 1:] |= majority[:, 1:] != majority[:, :-1]
    edge_mask[:, :-1] |= majority[:, :-1] != majority[:, 1:]
    edge_mask &= valid

    rgb_f[edge_mask] *= float(edge_darken)
    rgb_out = np.clip(rgb_f, 0, 255).astype(np.uint8)

    Image.fromarray(rgb_out, mode="RGB").save(out_path)
    return out_path


def _collect_detailed_render_jobs():
    """
    Gather the full-cloud prediction outputs that should be re-rendered in the detailed style.
    """
    jobs = []

    if FULL_OUTPUT_SUMMARY is not None and FULL_OUTPUT_SUMMARY.get("full_pred_memmap"):
        jobs.append({
            "model": str(best_result["model"]),
            "strategy": str(best_result["strategy"]),
            "pred_path": str(FULL_OUTPUT_SUMMARY["full_pred_memmap"]),
        })

    if "EXTRA_FULL_OUTPUTS_DF" in globals() and EXTRA_FULL_OUTPUTS_DF is not None and len(EXTRA_FULL_OUTPUTS_DF):
        for _, row in EXTRA_FULL_OUTPUTS_DF.iterrows():
            jobs.append({
                "model": str(row["model"]),
                "strategy": str(row["strategy"]),
                "pred_path": str(row["full_pred_memmap"]),
            })

    deduped = []
    seen = set()

    for job in jobs:
        key = (job["model"], job["pred_path"])
        if key in seen:
            continue
        seen.add(key)
        deduped.append(job)

    return deduped


DETAILED_RENDER_JOBS = _collect_detailed_render_jobs()
DETAILED_FULL_OUTPUTS = []

for job in DETAILED_RENDER_JOBS:
    model_name = job["model"]
    pred_path = Path(job["pred_path"])

    top_out = FIG_DIR / f"full_{model_name}_top_detailed.png"
    side_out = FIG_DIR / f"full_{model_name}_side_detailed.png"

    rasterize_prediction_memmap_to_png_detailed(
        XYZ_MM,
        pred_path,
        top_out,
        view="top",
    )

    rasterize_prediction_memmap_to_png_detailed(
        XYZ_MM,
        pred_path,
        side_out,
        view="side",
    )

    DETAILED_FULL_OUTPUTS.append({
        "model": model_name,
        "strategy": job["strategy"],
        "full_pred_memmap": str(pred_path),
        "top_view_png_detailed": str(top_out),
        "side_view_png_detailed": str(side_out),
    })

DETAILED_FULL_OUTPUTS_DF = pd.DataFrame(DETAILED_FULL_OUTPUTS)

if len(DETAILED_FULL_OUTPUTS_DF):
    DETAILED_FULL_OUTPUTS_DF.to_csv(REPORT_DIR / "detailed_full_cloud_outputs.csv", index=False)

print("Detailed full-cloud renders:")
display(DETAILED_FULL_OUTPUTS_DF)

render detailed side: 100%|██████████| 19/19 [00:04<00:00,  3.91it/s]

Detailed full-cloud renders:


,model,strategy,full_pred_memmap,top_view_png_detailed,side_view_png_detailed
0,HierarchicalExtraTreesTunedContextKNN,KitchenSinkGroundSpatialCleanup,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
1,XGBoostContextKNN,KitchenSinkGroundSpatialCleanup,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...
2,XGBoost,KitchenSinkGroundSpatialCleanup,outputs_multimodel_kitchensink_learned\predict...,outputs_multimodel_kitchensink_learned\reports...,outputs_multimodel_kitchensink_learned\reports...


: 

## What this adds, and what still remains outside scope

This notebook implements a strong classical stack:

- multiscale grid context
- terrain support features and a dedicated **ground-proxy surface**
- fixed-scale weighted covariance geometry
- **adaptive** geometry neighborhoods chosen by omnivariance
- tuned ExtraTrees variants and XGBoost
- optional **contextual kNN probability smoothing**
- **spatial / blockwise CV** in addition to point-wise CV
- optional **connected-component cleanup** for tiny building and car islands
- optional GPU routes for XGBoost, cuML random forest, and cuML nearest-neighbor search when the environment already supports CUDA

What is still outside the notebook:

- a true upstream **CSF / improved-CSF / object-based ground filter** package. The new ground proxy is a practical in-notebook approximation, not a claim that this notebook fully replaces a dedicated ground-filtering library.
- a full object-segmentation workflow with segment features learned before classification. The cleanup step here is intentionally light and post-hoc.
GPU note:
- the notebook still does **not** force-install RAPIDS, because those packages are CUDA and platform specific
- the safest easy GPU win is still `USE_GPU_XGBOOST_IF_AVAILABLE = True` when your local XGBoost build already supports CUDA

